# Required31B writer reasoning-budget comparison70
V21 writer4096 versus v23 writer1024; every other archive byte/configuration is preserved.
Twelve counterbalanced executions on the same six exposed CPU63-qualified cases; no holdout.
Original SequentialAgent, planner256, output8192 and shared three-minute timer.
Four native CPU compiler/wire checks pass; no model repair gain is established yet.
Private offline L4x4, exact31B QAT revision2;75min planning guard. Preserve all outcomes.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
SESSION_STARTED_MONOTONIC = time.monotonic()
MODE = 'gpu_comparison'
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not IS_KAGGLE:
    print('Definitions only: local execution performs no official inference, fixture grading or installation')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
"""Unpack exactly the archive and members declared by the frozen study."""
import base64
import hashlib
import io
from pathlib import Path
import zipfile


def unpack_frozen_agents(run_root, plan, packages):
    for arm, package in packages.items():
        payload = base64.b64decode(package['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != package['sha256']:
            raise ValueError('Candidate package differs from frozen bytes')
        expected = plan['arms'][arm]['files']
        destination = Path(run_root) / 'agents' / arm
        destination.mkdir(parents=True, exist_ok=False)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            names = archive.namelist()
            if len(names) != len(expected) or set(names) != set(expected):
                raise ValueError('Unexpected candidate package members')
            for name in names:
                relative = Path(name)
                target = destination / relative
                if relative.is_absolute() or '..' in relative.parts or not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Candidate member leaves the submission directory')
                data = archive.read(name)
                if hashlib.sha256(data).hexdigest() != expected[name]:
                    raise ValueError('Candidate member differs from frozen bytes')
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
        (Path(run_root) / (arm + '.zip')).write_bytes(payload)

PLAN = json.loads('{"tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "guide_sha256": "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb", "expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "timeout_semantics": "Per sandbox command including verifier pytest; no whole-verifier deadline", "supplemental_dependencies": {"http_manifest_sha256": "298380d8514f947567daee5b1d9730844b432acc01d5bf92ae12d3ad0a4799bd", "snapshot_manifest_sha256": "fa8dbf0fc031b15fee87e3af60b5d0eccebc4f6780d5fe84e7d73645dcd3ce50", "http_profile": "httpbin_legacy_redirect", "snapshot_requirements": ["inline-snapshot==0.36.0", "asttokens==3.0.1", "executing==2.2.1"]}, "events_compaction_config": {"compaction_interval": 5, "overlap_size": 2, "token_threshold": 14336, "event_retention_size": 5}, "wrapper_compaction_interval": 5, "context_cache_config": {"min_tokens": 2048, "ttl_seconds": 1800, "cache_intervals": 10}, "sidecar_verification": "Canonical frozen bytes equal explicit task limits; actual constructed EvalConfig recorded. Stage1 loader and environment precedence unverified.", "created_utc": "2026-10-08T05:29:06.331443+00:00", "scope": "Exploratory v21/v23 writer thinking-budget comparison on the six already-exposed CPU63-qualified cases", "tasks": [{"instance_id": "fastapi_14430", "repo": "fastapi/fastapi", "base_commit": "cff2236dacb7aedcaf2d2de0873cfbc9bb2b897a"}, {"instance_id": "fastapi_14953", "repo": "fastapi/fastapi", "base_commit": "d2c17b603d2f986d5f3800d3bf24acba8e1f485b"}, {"instance_id": "fastapi_14482", "repo": "fastapi/fastapi", "base_commit": "71a17b5932e38bcf798fcd05c5bb384e6d4db5bf"}, {"instance_id": "fastapi_14361", "repo": "fastapi/fastapi", "base_commit": "5265c4f5cb4d8fb4fb5015bccf1a6de8f3fc6044"}, {"instance_id": "rich_3882", "repo": "Textualize/rich", "base_commit": "4d6d631a3d2deddf8405522d4b8c976a6d35726c"}, {"instance_id": "rich_3506", "repo": "Textualize/rich", "base_commit": "4101991898ee7a09fe1706daca24af5e1e054862"}], "selected_problem_only_input_sha256": "711abc3a0494271c6e65c193f69f8a3f31b0e7eed0e2ed8105e6f64995c84453", "schedule": [{"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "fastapi_14430", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "fastapi_14430", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "fastapi_14953", "trial_id": "paired_r01"}, {"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "fastapi_14953", "trial_id": "paired_r01"}, {"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "fastapi_14482", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "fastapi_14482", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "fastapi_14361", "trial_id": "paired_r01"}, {"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "fastapi_14361", "trial_id": "paired_r01"}, {"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "rich_3882", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "rich_3882", "trial_id": "paired_r01"}, {"arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "task_id": "rich_3506", "trial_id": "paired_r01"}, {"arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "task_id": "rich_3506", "trial_id": "paired_r01"}], "baseline_arm": "tool_paths_v21_hypothesis_pipeline_budget3_v1", "candidate_arm": "tool_paths_v23_writer_thinking1024_budget3_v1", "arms": {"tool_paths_v21_hypothesis_pipeline_budget3_v1": {"files": {"agent.yaml": "83347c80e6f281774d6923e110fb10f6532bc8dc6cae058886abd952ff47e2f9", "configs/planner_sampling.yaml": "4f07c21d9278bb18102dc13497bec81e9f52b932c26cf0d75ce279b73375e5d2", "configs/sampling.yaml": "2a47816e29420420ef12abdefe80787c8759f096dcdc2d6b52307dbad33cb971", "eval_config.yaml": "f6ed0e7f68df70056e6646df2e2cbe6e3af27686ae0991c26f2ff8e192856783", "planner.yaml": "b27631fde874c029a8345c27ea6ac38b7bc180046f0060668e8e8167818eae6c", "prompts/planner.md": "85ea8411a708049eee9a3c9317c8045ede88244a4e29144fc4cfb572b1dad4aa", "prompts/system.md": "c5444cf541de7838803be8ab85309acd7aa4f5d0d5e0563075464d31f96b910a", "skills/source-import/SKILL.md": "8060329ada9f11b74ad56dd403165ae5c1f8c50396b115056e0c031235f0efcf", "skills/source-import/scripts/source_import_guard.py": "482b8dee8690cfe91dbbba19758a89376fa362f86d75546c84317b29aa5f8120", "writer.yaml": "dc11cfdd6a3a9dae9636a618ec1ee009d3862fd6598b52a176db4c7404122f7c"}}, "tool_paths_v23_writer_thinking1024_budget3_v1": {"files": {"agent.yaml": "83347c80e6f281774d6923e110fb10f6532bc8dc6cae058886abd952ff47e2f9", "configs/planner_sampling.yaml": "4f07c21d9278bb18102dc13497bec81e9f52b932c26cf0d75ce279b73375e5d2", "configs/sampling.yaml": "22bb782c00d19e4e26034ef2d1668e6980c892691db731f90c737590db0fcce4", "eval_config.yaml": "f6ed0e7f68df70056e6646df2e2cbe6e3af27686ae0991c26f2ff8e192856783", "planner.yaml": "b27631fde874c029a8345c27ea6ac38b7bc180046f0060668e8e8167818eae6c", "prompts/planner.md": "85ea8411a708049eee9a3c9317c8045ede88244a4e29144fc4cfb572b1dad4aa", "prompts/system.md": "c5444cf541de7838803be8ab85309acd7aa4f5d0d5e0563075464d31f96b910a", "skills/source-import/SKILL.md": "8060329ada9f11b74ad56dd403165ae5c1f8c50396b115056e0c031235f0efcf", "skills/source-import/scripts/source_import_guard.py": "482b8dee8690cfe91dbbba19758a89376fa362f86d75546c84317b29aa5f8120", "writer.yaml": "dc11cfdd6a3a9dae9636a618ec1ee009d3862fd6598b52a176db4c7404122f7c"}}}, "frozen_candidate_archive_sha256": {"tool_paths_v21_hypothesis_pipeline_budget3_v1": "ac1f7acd322707b6ec735fcbb3e3caaf5c2b5551a686d57f9b9b755ca068aa90", "tool_paths_v23_writer_thinking1024_budget3_v1": "7299ebf74d5d7c7972652a0c47af5c09d9811d16d39c0858b1a75c02cac3bb30"}, "limits": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}, "proposed_model_limits": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}, "arm_limits": {"tool_paths_v21_hypothesis_pipeline_budget3_v1": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}, "tool_paths_v23_writer_thinking1024_budget3_v1": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}}, "notebook_walltime_budget_seconds": 4500, "changed_factors_between_arms": ["Only writer thinking_budget: 4096 to 1024; one changed member and one scalar"], "unchanged_factors": ["Required31B QAT revision2, original sequential planner/handoff, prompts, tools, source-import skill, output8192, planner256, other sampling, shared3min/50tools/100turns/cmd300, cache and compaction"], "hypothesis": "A smaller per-call writer reasoning cap may reduce long decode intervals and increase verified patch delivery inside the unchanged shared180-second timer.", "causal_scope": "Within this repeatedly exposed development panel; native request transmission is verified, GPU enforcement and repair improvement are to be measured.", "native_candidate_two_repository_integration_passed": true, "native_candidate_reconciliation_sha256": "170e4ad0e66909666f0bde6fc774199019e99af28bde16e24c996b166b4cca76", "native_controls_are_not_model_repairs": true, "hard_per_phase_deadlines_claimed": false, "shared_task_budget_includes_planner": true, "CPU63_reconciliation_sha256": "20726a8141bbd387b8467756aefadcd52c023617474e15133a1a6f15edba24d4", "CPU63_plan_sha256": "6ad805f871c4f73d662ecc40dde7f6d6b282fbf4f0de05e179d561bb3feb19bd", "CPU63_analysis_sha256": "0a8131b3e4c794e9cd132cae77e52f32517b28b43d44418782611ccb0f9cbb83", "qualification_denominator_cases": 8, "qualification_denominator_checks": 16, "qualification_classifications": [{"task_id": "fastapi_14430", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "fastapi_14430", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 9.14260772800003, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "84d52a9dff2a4ae0c2335ce4e6ab54a6e30d48c00894dbeb02925eea2ea77228", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "fastapi_14430", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 5.147027769000033, "replayed_patch_sha256": "0592f55eaacb20e012f1772d6bc0594e26a3924b190293019ef6951147ce3c56", "replayed_patch_bytes": 1057, "test_output_sha256": "f1fcff76bc96cb6c5ec3e03f8cf65fc0497bc9da196e55c1cefb1b039c58c91d", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "fastapi_14953", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "fastapi_14953", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 12.451916418999986, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "9aa5a6c90f8e6f086dfa3796ba26a0ec48f6e4e9fabe79890c6644d19a76eb92", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "fastapi_14953", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 7.198212582999986, "replayed_patch_sha256": "ebb0f3c6dd0e979fb7e77f683306303a666970c7ae824f08702343c8223873c9", "replayed_patch_bytes": 4315, "test_output_sha256": "4d78fcc0ba7201db1693e89974da949ee6457a6f3b05784bcb6a5c32e872651f", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "fastapi_14482", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "fastapi_14482", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 8.456928779999998, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "310d0ffd6307fd4fbbe0d3c0c0f4e0c45b65cb2d928d03bd862fd92caea93ad7", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "fastapi_14482", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 4.994778461999999, "replayed_patch_sha256": "ef4af9a1bc6941e7b7f9d8ce88545a13c74632da90df2152c8ff5458ef20ac5d", "replayed_patch_bytes": 2520, "test_output_sha256": "c0970fe1bfdcb51ccfe5a979285b87fa31d1879ea87a4904f41ac90bc4c0b37f", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "fastapi_14361", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "fastapi_14361", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 8.267683924999915, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "05eff1c9ba2eceafe484c393e7e78c628d28d7d174fad8491a863f717e6cc4b4", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "fastapi_14361", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 4.955905084999927, "replayed_patch_sha256": "e090e89ccc3f1c34bd955726f2add87ed0ed7b83e9f10b5137e0f0cdcab5bd60", "replayed_patch_bytes": 462, "test_output_sha256": "e2c682cfa69f25cbf90c31e4068109e4c860a85031fd92dc169d67a53a9ecb5c", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "rich_3882", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "rich_3882", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 5.023496830999989, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "7f5026cc682b9fa924960667a821263ea248edfac92618d61d02a8e42e757041", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "rich_3882", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 3.7365717260000793, "replayed_patch_sha256": "7ecfc850b865ae7775eb0d7d289ca85102b699b990b55304392a3ba740e0878c", "replayed_patch_bytes": 459, "test_output_sha256": "ee22dc53e7435e0a58f286bc52375ce61e2b2c9dd804c6224af3a9401d10ec9a", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "rich_3506", "classification": "clean_base_nonpassing_reference_verified", "clean_base": {"task_id": "rich_3506", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 4.993925839999974, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "3d38cb7eed53d428b8cce92b0c705681042fabd6e851cbf3652991d566981e17", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "rich_3506", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 3.8335210440000083, "replayed_patch_sha256": "55cf396bffac014286ef76f179d3fa79b34eb1b2d42a4d0bc95109b8e199bd0b", "replayed_patch_bytes": 1082, "test_output_sha256": "5c1be5b8e7bcffa427cc9077dced2adee42b4c8a8a556dda26f08f5fad8e0bae", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "rich_3130", "classification": "official_reference_not_verified", "clean_base": {"task_id": "rich_3130", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 5.879339466000033, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "e5bd2827e919d2afd2e2812cf523e5239f023b01ee4bafbbf070a37e706b71c6", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "rich_3130", "condition": "official_reference_patch", "status": "completed", "resolved": false, "test_exit_code": 1, "duration_seconds": 4.481619544999944, "replayed_patch_sha256": "f51a83ea4fed0d343eee89def13265b14819fccc387cd856e2968e08d11e2384", "replayed_patch_bytes": 1215, "test_output_sha256": "ef93340e47148964e6f79593ccd0cf3004f7a4f8ff2fc29b40dd41de412d943a", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}, {"task_id": "rich_3480", "classification": "unclassified_nonpassing_clean_base", "clean_base": {"task_id": "rich_3480", "condition": "clean_base", "status": "completed", "resolved": false, "test_exit_code": 124, "duration_seconds": 311.04295295400004, "replayed_patch_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "replayed_patch_bytes": 0, "test_output_sha256": "92a7d63aa02284782b9ac99f672a8cbd9dcfe3ff8014df42bc57a7aa2f127768", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}, "official_reference_patch": {"task_id": "rich_3480", "condition": "official_reference_patch", "status": "completed", "resolved": true, "test_exit_code": 0, "duration_seconds": 11.060410937000029, "replayed_patch_sha256": "223da917b0a08a8e15cc89fc3d5e5ad5ac1f12e56742229acd9da32b869bfce1", "replayed_patch_bytes": 810, "test_output_sha256": "c51e895a1ac7411e3d045ba155447161018fc726e63b2507270e75b0d177a289", "captured_diagnostic_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "reference_and_test_text_exported": false, "agent_generated_repair": false}}], "repository_qualification": {"Textualize/rich": {"assigned_cases": 4, "classifications": {"clean_base_nonpassing_reference_verified": 2, "official_reference_not_verified": 1, "unclassified_nonpassing_clean_base": 1}, "qualified_case_ids": ["rich_3882", "rich_3506"]}, "fastapi/fastapi": {"assigned_cases": 4, "classifications": {"clean_base_nonpassing_reference_verified": 4}, "qualified_case_ids": ["fastapi_14430", "fastapi_14953", "fastapi_14482", "fastapi_14361"]}}, "selection": "Every CPU63-qualified case in original assigned order; two other assigned outcomes retained, no replacement/refill", "repository_coverage": {"psf/requests": 0, "fastapi/fastapi": 4, "Textualize/rich": 2}, "prior_GPU62_outcome_known": {"v15_verified": 2, "v15_completed": 5, "v20_verified": 1, "v20_completed": 6, "unattempted": 1}, "all_prior_model_outcomes_known_before_freeze": true, "parent_plan_sha256": "af4aeacfa1d2d579257eb154b6bb891fcdb712abe8e9d04d34148f5d616a97b8", "prior_GPU65_outcome_known": {"v15_verified": 1, "v21_verified": 2, "completed_slots": 12, "v21_only_verified_delivery_cases": ["fastapi_14361"], "unknown_verifier_slots": 9, "holdout": false, "candidate_uploaded": false}, "support_review_policy": {"minimum_candidate_delivery_advantages_for_main_review": 2, "baseline_delivery_advantages_allowed": 0, "all12_executions_required_for_positive_review": true, "all_verified_candidate_patches_clean_production": true, "no_candidate_protected_test_conflicts": true, "unknown_grades_never_reclassified": true, "native_validation_and_separate_runtime_feasibility_review_required": true, "operational_review_policy_not_significance_test_or_prize_prediction": true}, "notebook_allowance_basis": "GPU65/GPU68 completed the same twelve-case schedule under50min;75min leaves25min overhead margin. Planning guard, not billing or global12h guarantee.", "research_assistant_public_pool_reference_exposure_recorded": true, "reference_patches_exposed_to_agent": false, "grader_assertions_or_reference_patches_changed": false, "held_out": false, "sampling_seed_fixed": false, "holdout_or_confirmatory_claim_allowed": false, "stage1_sidecar_loader_executed": false, "whole_run_twelve_hour_compliance_established": false, "full12h_scoring_schedule_validated": false, "hidden_task_count_known": false, "private_scoring_equivalence_established": false, "public_official_baseline_score_known_before_freeze": 0.08, "main_submission_selected": false, "competition_entry_submitted": false, "official_leaderboard_score": null, "agent_generated_repairs": null, "USB_testing": "deferred_at_user_request", "stopping_rule": "Twelve fixed assignments; preserve ordinary agent timeout with no graded patch and continue. Halt on unknown -1, raised infrastructure exception or original walltime guard. No retry/refill.", "analysis": {"primary": "Verified paired repairs with candidate-only/baseline-only outcomes on this panel only", "secondary": "Per-arm original compiled generation, native planner/writer authors, per-call completion tokens/thinking message sizes, response-to-next-call gaps, edit timing, clean production diffs, protected-test conflicts and patch submission", "missing": "Retain every fixed assignment and unknown grader outcome; no imputation, retries or case refill", "trace_limit": "Native root system_instruction is empty for SequentialAgent; static compiled child instruction hashes and actual events are retained, full rendered wire is not captured in inference"}, "expected_generation_by_arm": {"tool_paths_v21_hypothesis_pipeline_budget3_v1": {"planner": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 1024, "thinking_config": {"include_thoughts": true, "thinking_budget": 256}}, "writer": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 8192, "thinking_config": {"include_thoughts": true, "thinking_budget": 4096}}}, "tool_paths_v23_writer_thinking1024_budget3_v1": {"planner": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 1024, "thinking_config": {"include_thoughts": true, "thinking_budget": 256}}, "writer": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 8192, "thinking_config": {"include_thoughts": true, "thinking_budget": 1024}}}}, "native_candidate_compiler_and_wire_checks_passed": 4, "native_candidate_wire_requests_verified": 16, "native_candidate_integration_is_not_model_repair": true, "prior_GPU68_outcome_known": {"v21_verified": 2, "v22_verified": 1, "completed_slots": 12, "unknown_verifiers": 9, "v21_only_verified_delivery_cases": ["fastapi_14361"], "candidate_uploaded": false, "holdout": false}, "terminal_interpretation_prepared_before_enqueue": true, "actual_GPU_thinking_token_cap_enforcement_verified": false}')
PLAN_BYTES_BASE64 = '...'
PACKAGES = json.loads('{"tool_paths_v21_hypothesis_pipeline_budget3_v1": {"base64": "UEsDBBQAAAAIAAAAR126OBSzkAAAANAAAAAKAAAAYWdlbnQueWFtbGWNwQ2DMAxF75nCC5QBcmOGDhAZMMWSsdM4AWX7QqWeevvvv8NT3CmCUisoyU9KmTMJKwV8kdY0C7pHeNK7Xcgo432HhXwunCubRhghFzvYL0ABdm8EW89WN3J2WE3ETlpg6qBY+SAolM25Wun3RC5D8Dalb9JjAHjAbLryK2WsW4QsqEpl6LjLvz0L15/8AFBLAwQUAAAACAAAAEddtNr/QmoAAACAAAAAHQAAAGNvbmZpZ3MvcGxhbm5lcl9zYW1wbGluZy55YW1sPcpBCoMwEEbhfU4xJyhRVOhcJlgdk5CaCek/0OPXdtHdB+9BziZ9hXVh8rfBQVtoX97nnwvT5N25voMamiFAi9QX0+DHySHlWnKNYdN65MiOKNftabsEJLWYcJ3oJlf4vw/bo4BpnBf3AVBLAwQUAAAACAAAAEddmFcxJlQBAADvAgAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbK1Sy24UMRC8z1eUNOcsSYCQXcQ5FxSQEOdRj6d3prXjh9ztsMuJj+AL+RLsACuR85Zk+WC7XI/u8eGS6Hq4GPYy6ysln1YJ8+ZEfsWvHz/xwIEzmcSAz5TJs3HW+uLSCr4GK4EnqFG2qgDK1nbd4DEikbnlSotzrArxKccn9hwMCylG5gDPpCXztLm4uM7YpxZCpd/henODM3p8+ZsYMnuSoAgxTC0kL0HUxOGb2BKLgbCXY3PIPHUW05Aa2fYt/kePx+JWLop/ZTz3cGBO8FENcY9qf6RRVrETPKk+0x12eHONl+jxUbwYnqKjsayUT9CUmSbsY67LFa2a5nPLnafjUOWmYoPFAwfd4f5me9uYPuWZgnznjLnIxOBjU8jvYdFobUNkfLRzEq9v393dd7ZIOFQXw58Z23WABLeWiYeWy7xY/cFy4XpwvjuWaWZrhrZ33W9QSwMEFAAAAAgAAABHXen1ArBMAAAAXwAAABAAAABldmFsX2NvbmZpZy55YW1sSy1LzClNLMnMz7PiUlAoycxNzS8tiS9OTc7PSym2UjA2MAAK5yZWxJfk5+fEJyfm5ABFTeGCQPXxuZl5pSWpIMUw0dKiPCDXEKgXAFBLAwQUAAAACAAAAEddk6/TBMYAAAA7AQAADAAAAHBsYW5uZXIueWFtbHVPS2pDMQzc+xTuAZwSGrLwrvveoBSj+ikvprLkWjLh3b4OCSGbroQ0o/kwVIy+qA5M562JnVGLOliRLWUC1eg/qL5fd1dlQYp+xVohHMLb/jsUC79g4XKA/TFkc4XV+shWhKN/KZxpLOhbl9pMXxsBM/ZdXdwdSlnYpva0YWF0MqwNSz+4Rd+xQenPsUyEJvPzy8082MEe/9d5KuuT5+3w8EwKtVHhdbdBJbcUBSK5JOvAepq4SWrQp1T0swH+w0Dseif8AVBLAwQUAAAACAAAAEddm3tnxF4BAABGAgAAEgAAAHByb21wdHMvcGxhbm5lci5tZGWSwWrEMAxE7/4KfUAI7aWHnkthr6WXnooTK7uithUkJbuh9N8rZ7ew0Jsx8szojV9ZCkTQE4t1MAuvpMQ1ZhCcIwmctpnthEoKk3ABPwOpLggDZj738MELnOKKUDkYc1aINf3dGFDVGUfDBMqLjNjDG9oiFeYcqYLhxWCpCQUenx7gzJIURl5RqB6fQ3PDy01gQFcllm4PkekL8wZU5owFq0Xz3FBILX5hB9GgsBrYmV0hjhYo+RRNhPJvFeO/nN2enisCD4qyxiGjozgKauPi0Wqi5tSHw3SnkBj1uu/VZWtQtzJw7iChjkID7uOCGddYzQ86c1UaKJNtfXjhG6/VBWCijDUW1M4HJxSsIzrBvDRvvzTUXWLJpsDibiM3Eo3TRJc+vO9ee4Xx2BTPlDM0rp7NHyTSMUryTN7sXcnx6LW4thNb/BfcSgvh0NZ8Dt/+RRxJ+bwuNbc4P+EXUEsDBBQAAAAIAAAAR12EnRvPUQsAAF0aAAARAAAAcHJvbXB0cy9zeXN0ZW0ubWSFWU2P5LYVvAvo/8BbbEDdi+Q4c1qvx4iRDyx2HRs+DdgS1aJHEmWS6plGkP+eqkdKYm9s57Q7kki+j3r16rF/dovS3qh/msXr4fNPTyf1yQQ3XI2KvVFhmefBmlYF18VXfmhDWIw635RpbbTTRdkYlDezCzY6fztUempVo4eB78JyHm18nnVs+pP6VzDKTcNNdp69u9oWO0fnhnBSH70JxuPYeTkPtlGNm6LXTQxpRzvOgxnNFJNZIw4wIeIrPo9GdfZNjHU+Yk+YF9ziG6O4NuLLcDpUh+pz1D7ybdOb5oUWXkx8DlHHJdSFF2p9BCt/MU1UdgrRL020bsJTbnpe7NAeqtFE3eqoT+rpTewV+8zbjFU0xPT6ap2vVaethGR/wl1wpHdt2vhQwWcGFWeozrtRtpJ4n9R3Fl+bNx6AqE3Rdtb4oF5t7JW/qK8u2EnZTi2TvuIkfR7M13D5k9GtbOPNYK4a4dsCqXlQLanCTvUeKZjYOWQ6Jxip0SHgNY63QVLeLQOy+OuiB5qBKEyuNWrSI0DjVDDaN/1zsCPs8M8N3j0CJTBBcBUQXGzfKllkxrNpW2waam4TAcazRSCRgwkpACaPg54ui74YHGi8NSEB6eL13KvJ2Et/doxEbybx9LxcVONdCAbQGRFcAOVRXbG0u+VVsGQZ4Ki+aCZWNYv3hFZnB27/D8ScbjpPHAzrWUzzoWodPKChszdXLnIe5tPejDjkfnBeontS76/OtmrIcaN5r70bTAE1weUTzgHqQ6/0BpQSGsm7zuhgkdiT+j5BIFWSd7Cm0UvIaO/NoXr/8fsNaXiioxqXwBSM8FiqBgbKVoKQBKw1DNG8xRIEz4wL7GTYX72NRh4ovIe5v2BfQQFy8SoRVPK8NYM9G8SBzsoqmD0F1oVY3dquO1SdGwb3Snd5EGL/rZPgwlBHCiIea/Vq9Av8BwyNzyXYa98eG0HQWmt6Cq9EsmSdwQHgh0USgfrtvE4lvHhT00Sbjelti2gqfGvb9LHUM9PyrQ0NzPBiMCIEwItJyi/ThMdSpCtFoHg7e1nWzH9aEhw71yAzqBFvUKMAFg5gmiZUCYgoVVxnfYgn9Y13mrYIO4XElajp89KCpmQV7JlNjjm4FgeMWOYWvBocoojMjvguE0O0o3FLTOjZeKHgWlR0zFz3oFqXKhABMYJCRIpk0JN6NNi1aWA/al9FvHzXEziznZHnCQVO+PEvLmR4Gz0z1GLjalY+SkrENjYONyQ81cYi4AfHTFfr3USCehdMXObVkhRsXcYRi81sJkSsAS1IG0NRsqLJae1aDSrgP2f3BipVrutoLmKW4A1EyPGH6uMt9m4NfY1oZuJMVX0EbaK1qPBihyExN/LL1gdq8i1S1aSCkOrLUMmEkJeSHnO5cZ3z9mITFAqfMzWhVFCx+la0BE3Kzb7tXfTV+Zcw68ZsMMweKtkbLrsBJQj4gauuPB4kCgAfj5qFd8zegQ16NmKc25GqhS0lptOWrKJCDlVpMo2CV2yWko7cAckoHhn0k2Gf+3WxIP1n4v05w+EZC9C3ARGnnn58+vSzKpPwmGgCgEJAchDDymDL1PRoC6Y9QbmwQlv3OjFnhwpJyJ0Gi3d4gIvVPaBs4nHNjkFY7XT7P9W7U8/Ki8yb7mLmhpX6kapDBfM6WNCv3mBfFo4ROSGFt8qmFKR9bwHDXpIn9UMvzac14hPbZuB2MRGeUJH0LK4UrOwMpL6yUzMsbK1qvmV0nKIbE3z5Ldef5tvXj6KeCj/WziEk+IHJKHRSdrtoc3V+pNVo9IQDyRGMRV2EaUcPdi12YydkYmOPfEQdXv4EgZB5MhFfzZ4JdiGbSUfm52lTSSj2Y5xI1KwNNjcTbdw3YKb//BfVw7mg9qgkNGjRCooreXrIfS5ppiwnV3towSOglnADYQbB85uu4PHgzqCAZIAIscS43IIiae0+woqikreuKEWIwtMzsqYbypXZsIhR88QuO1xCvyzpb7PDvwFOLmFVGezFhvoavLSqkHZhIbMhlxKFKf4ONSPtA8KLMHGUWbV6MdCUIP6e9NVlUpPanJ2l8lliYK9YY7VOAF+RCIFtKk31Lo7zu0mmi2f05+d9/Wlsv34o1Eud64GIfhdu49kNsKrRqeODc6gQwIotvrzvc0XN1KQGlBNIIt7q3GkhZ/Ra2OboScEJ6aBqwVEKp3BE4R8iqkHbV3tJFaUtSI6pl8LbwssI5nbS2Ql9DEmotwRfYBqzCsqVrcWk7eFfn95/W69qJ7G6IdNpmaTACgx7UDdMaY0XECSfqNrvpitiKrfcUq3Iu3oHPTo8kJHLPMvd+22MHyFn4WEihFWxZKpjjYLRH9gkRFGeEUNY5+k61RGGuRzrjRVgaVH8BHehtaxUZJ7i6rUUKB87M4naTT5Ioi06TyGGdNOQ7MyQ5jVqa4svs3ryelqbQzNoO2apNZNuW9kO+D1UhazB3/MSCQIUnqhpCko44NBPN92/mZimWmb/0z44rmOrGCgxOSb7W2hJO7M1Pxyqn0SQydyDYWpqUoO3RKwtNPI2cLHNd+jXeHvdKgImNQs7sBHuTwVRqzTyUG5wJc3IEzW+9DdB9jbiNb1zwhn6Tr1jeoBfWYrjSHb5Qkx8MUOC9T1zcDZ5CEnj+RezrSiVk/oxpf5L0sApEbpOb21wXiXq3TidVMC7D3//XrqdRBpdkpVRTr4YuCYgQIeSDJJEJaFsymwt4dQ+J8EKJAvY70rjOvjFt3B7SVM5YpPHhf1O4YPUJfMLZ+T0u8ktNN7OMWQ8vEmBQsRTXEG1S9dcJgtVJkyJ+RIyiQwMKJT8yvBFb3blWlbtPdvkbnQ8CnGDOAsK4rv6nn42QhrCMYmJ45ENxfM/5o3EYbhuaiF0URxNg3EjcnQDySNit8yeSW8vkpQX/EVNSb7dqU14ooiTnEb2Y4IO1T5JFuHjiFdvgnu9kkptucSMFD1L9x51mWnMOkyJ6DK+saG4nNlaOqZEshxkQJKQkvU0eNiYK123R9mq4DDx4yFftmw1m+hm1Vq7LmPcttE8O8Zh8v6K6UxuGoxOFTDugpfCPoWB+Y8p0Lxd+7iwSG9SyK3Vl8nBjkaJHF4nJtCdwFGmgxJ3aoPd73T1rE1NswDrpL90dZLGu/LiTCoLLH93X1ckdHN9HQl4HbdddD3ygkVysSqcfdwO9aZaV+jIJdwX8eUQxk/22wIZMEb9QmpKPZhN4P9U0h9A+1Ct2BZJrr7Ag1RRus+Qxk3FAvHlkzazu7zlpuleKoWxhH0SJHncKSb2orcXqgdTnzJ67X1AavUDGzHNTBAsqm5TFQ8im/cJUsyW21hFiSSXPPV+AST7FNc/bk63O7B2z79EJt/QrZQFwYW2Ya/beJHMSjcUEK5mvQLODSdwhhRQyodM3m8ZmQ4BNNZxUiRQIuPfwzbPrMYXaqyaZ4OeeyOtKn2TZIsC+Rk0V7kmykMcG+9LAuv9kVXzh/yfGWATlSKr/N1dmMw2q+qrEgWne93iIPSyV1h6Up9JCenqeNnmlDxl744ycqS1qphU9H6KvE9UtWsB+pEm01NVcQTZg40jOBfIfQD44ei645pd2areytXdWZ1vRKrAq+HfwOCWc2kS+r5z8hKpEOMT9uZeBdmwY8iNMwwP3a20+KT+xuml2C4VZGbPArESXWCgMG+nz7DRoaXQEP2L3dOFiafO0vIDCdPq3XKhppRaZhMpdDcOlJvE2yqKYY/3mLCr6iN/CAkiABMHpv5SjHXbbwFyTciSv5iH6t/wQVv/vH/4n0oMtJKmZcqXOG0ameUeQMYulSb6ncRzXT6CSL2XXzvyleGmjxAbjR5i8pWEpPN+SFfiHsAEW5KF4VT9F1BLAwQUAAAACAAAAEddSq/8XFoFAAA0CwAAHQAAAHNraWxscy9zb3VyY2UtaW1wb3J0L1NLSUxMLm1kXVZNj9s2FLzzVxB76cXSHtpeEvSwQdImBZJdpNu0QVEYtPRkv65Eqvzw2v8+80jR1uZi2Cb5PubNDNk0jbJmolc6uOQ7anianY+qp9B5niM7+0p/Ic/DWceDifrhHA/O6u5A3VPQZTdWSFPPkXr97PxTmE1HS0BtbK87Z7GKWGYcz5ptiPhS4s3e/Udd1G4YRraERc0RgYMbjcQLOL9zJ032yN7ZiWxsVYOq1RsanCcdfQqR7V6bWlukEDe659C5I/kSrtQpnerBuylXbLqYzFgrUKt6d4nHXk8UTW+iafWfgbRPdhueeBy3BRn9zPGgyz857gsAN2rgkbazkT15f7gtG7Zlw3afjO/b+ZwTGr8P+p+bpplcn0a62eibDx8f7j8/bj/dfXx3s1FYIhuSz0tNA/jcc7NklCQ3/7b6wYSAaJpOXBBZ8CgxN9q6CJCAS/S8SzKODMhG5yqdV+FAGEvnpgk1tfoRGOGfGRhWMEMG7jeOqzlL/SlQWQqCRMmr2Ebysyd8ahPyHKyJfKTrUE/Upeh8qz+T6XOA3/+4/6Q9hTTGHBnNRJTUU6vUh1h5JzsLjqCIIC28MRgthYNe5W313egR+owQ3gvPCmaqHA4yv7kS0ggkrb5HcP/MGDkyo/lJCncWxDV6TzYJS4XsZpfT5qPKDaiJwxWWzYXRmSdNY13DtqdT+ZoZ1hSWyyDynz3NYSNdq5xOWrJXJYD/ma8rJZTYAkZh7M50TyR0Wpo2R8Oj1Nl+h53zvJfwe8N2A7gnDHdp0tIzPtEo+Zw22eglLEax36OFwW3UoumB/VTi1S09DwM++sy17mDsnlqN1JZEib17tqMzfdDoFHWS7ZjCRoGgPPDCIDqaMRmQAgj6JUbImg7yBzhOXiALCLyaGyrGBhMRLET90886kLhOeJ2DgtPzCEJcCCiy1Z3JdIeupiU5YIMvOTfeRp5I7VKPsK1+68p0nQXfLRlfMCdgaCG0Ii2PngQxIcqQOY8aYpr1gCFAuSDwnQWcIXUdhTCksfKci2xXc12OVM0exYAZ+O7oYI4MqdaY2ZqkmKpQscyrka0lGB3QdA77jR5ROjpEuXp0gEEJvlX5rzGo1QAvFp0nvk++4I1ZHLgvyS/VoMHlAlm6RChAK8EQ5SipAZNhL8eREtzkUBg8ez5KPXtvevI/hJeOf5cRFdllvGHGawsfXAcH6q/kWNsBEkslrf4VOb2ZuVcSB0rDGT1nr/pRN2908zd+dQYy2QKygU+/3MZprrad1bPNy0B47bMKLbnAoOxZAoBWVhq+oLPMo8rhAVWRP1KFu1gcsIgpiMkNqLxCnn3VINpBLrLVMEUJhUzz4oKjObuEew+ONMtRjioevEv7QyW98DpkWkjBy7U9Mvxtcb63Zez1fn4pUqk6vuwlXLOhWezU0C73hR9yc2YJB4myqCuPvqhKRX+WSwq0jMkXq7ODN7idEqbqL6ySvp1ggunme74OGWD9daBChGq21QEhqYnLflfUukaPCyWrtW5kg6LTPHLHtcem3Ir5CaQnU54tcksdy2vIZE9qsmc+fH18f//p4e7x/Yp46kI8GOBFBhVcY0G8pTxQrWiYBpP9oLxWXggAtLiCu10tbctdEeQhBMMw8zyei5GeoA5gLuhSj/n9nxj2sJWytwv3FqLLEN59eff5a30yYFIer4YuN18Ys0dPQX7W94G6B3PyO4OlraW6xUiz2q0Rv6QjC4ko+3Umy6oRnEMPUM5GCTq5buNHltuioNGsfXF5XEHwvjwfsyovzJ1SFAupL7qXhoUecZsHaLaY+TqutFhNbzQ8fedGWrBD1bkN9Q1QSwMEFAAAAAgAAABHXcZ82RunDgAAtC8AADMAAABza2lsbHMvc291cmNlLWltcG9ydC9zY3JpcHRzL3NvdXJjZV9pbXBvcnRfZ3VhcmQucHnVWv9zozYW/91/her+AGwxTTrd3k06zE0263Yzd5dksmmvvTSjISDb7GKgCJy4O/u/33tPEgiMney0vZvjBxuE9CS9r5/3xHQ6fdUkS1GLxGfFYpGluWCyaKpYsLIqNiKPcriNVyJ+z8SjiBvoye63LMpZtBR5zeT7NMuCyeRmlUom6yhPoqwAIos0E6yRQrKiStI8qrZMNvdAMxYSW9Ncpolg9Qrma8oyS4GuhNH3xWMwOa9ZLjaiYum6LKoaacAaonwJA2FmsYmyJqqLymcRq8RCVAJXWUZ1vPKxby1kzSKYp6rTIpfBZDqdThQtFlXLMqqkmCyqYs2SqBZ1uhZ6pvbZZ/j7G+zEjFtFcpWl9+ZR/UFD0NRpZlrfySI394VUU8CycKCZ4QoeTZeqpS5XNpmOU23Ltr2txbpE7rbPsNDJZJKIBVumtVsVRe2zF7BN6Z1MGFyVqJsqt4gGVZO7tw70dnzmzM7gV9YVjfT00DufxVEJ4wQvmrps6vCmaoRP9MYvXAZ0DY+PfKUv4XdRJoWnl5akMgZ9qvhDUb2XZRQLVzyC2OO0Di+AzXqt6YKZZpB0wlxcmGnx2Gchc75sKTgoa+Rn1yNIJU/SyvU0PbxiIJSiYCUL2W2//x31ErDQQ/2D+CFxgTU01l6At48j1NNIKgADw3u1sADUDy3nS+Z0hO5GZg/EYy3yxCVapccWsNsSDAc0KxD5Jq2KHCm7ztXPN28uL65Ob96gPB0vkLC72oVuqHtSwFjga+kpbQAp48YkjFQtSLedFel3S+iYAgTyou5etYw+6XEgLvI6zRvRNlZCNlkN86FutqNhmZXYzMgQSQdnclU8zOqizMDuM8ez51UkAqXGcQFeIwzZUX9e2lUQJZpbeoisE1DJIBE4ygW+1FVaogjgfZFtBGpXGivd9jyjgJnIyRYk6dtxN1EVpVKwH8H5iHlVFZXrnEU5cgU8GWx7sWXo+b4H1W0F+y2Yv5RsNrN0xjZKte6ygFVpQ1Hel2dFHKHz0va8LpIms4wEZ61EsGiybI1uz62c29PZv6PZb/zul4cX7t9OfgnsBu+FMyAyvqH5Yyli9PHoZh9TCcJcsqttvSpy48DyaG02AfIjXVKEtdo5gWP0qgELBhO6a9Usi7YgENQxvS/8RUuQVexYCwNGvYf4AoPViOBdkeaozO4LmrPTjx3ldc1YoMp5mqc150G5he3rF8FDWq+4bBaL9BEWC6+8gRIDf3t6jjY8VHSt26hFuMeu/5hq7YxsJagI4CSVyEDgG8HrQnni3fnGRfZWhWu5XUP0fg/aG23Acw31zb5IMEFUluhdzBp66k89nlb/c6UR60YObEBDCAIBIJMFbA19NUhZC7RvBTTd7dGdNoEY9pBzcHGG6XALPE6An27n+zzzKmhK5Lyr3ODry4ubf12f38xf/XwzP7t8PQ+dYxC+enlx+cPb+fVbeEutQ85cnV/xi0t+fvF6/pMeBi2vz9+evvrHnOP9jzj68oKfvZmf/R279PYBi9FbSHPAToBQtj379TG4kX352JdjQKyAcXYE/JydsgUIZQU6BXEUYJgARcU/pAhQKEsYKG1MzJWMIMYqTdBOq6hME8N7kaTgWYik8pkQGZwhePERsPgILupqC/Ovww7XqDuuVu5CnwCAweb2+M77FmSZLtM8hOgT1RCg1/6Uk5VwPvVpLxPttcMPU01HNnEsRCKS6QlBiamiMT1R//4UeCHB4U1PMOT3KOs3SHw69byPE/EYi7Jmc/qDN4D1GAbwA3MSGPGnAhWX19tSTE/w1xVewDm6NM71W7UA4d2evDw6uvs40bFEK5kCwdF9JkJkSffoK0FRq7r17yMpuNVsPXuTskrz2p2+vfzh+mzOz/95dXl9A5r34/zi5vL653D6BYomSJp1KXU4gwgFIiSJogtrgWBwI3C7ALFfpxVoF2qdntXR0WTZRFXCSaG44yG76N5yuIAMERQMMOJgixCoX1G4lvT7E/6CKhJZMyP7QtE+hBXpcmYxjEfdNOaBoPMhCZXFgH2Eto0QNOw9S/KhqG4E4Gy38dTkY9i2Q7AvBwgWR2BmRHiQUiSKZXgDzg0ZNkAapEcyBIxTZuSCVWgkGu7AH6PHhXaggIENBQsufY9SOBqvIprDWS1IBK76CF0sem+a52nv/R15GdBCUYHw6jbd6pwXW0SgX0nr5DSUI83MiiiRairw3Cb2hyDSYw+dBA1Rtg1DtBERXFWNPaK32jNJbhwkV5rr3MHg+6LIXE0KkwKCeOq5g3IeokIzekAbekRIE7a1IYrmjVJaaPqsa7OsVMNyg9Wog3bw9wAmIHzyaAMsQuOwg7Z+iT4XYHZT1rABMP8mzRK+Bs+mjBjvwKdEuB4Ng8otyPUdbCGoi3XmGGGbnmNgxGSB0B/T01bDi3yRLoG0fqHl1VKqRJTwGjIMV6ACQfgInaZezP7qeJ2C0ooR3RAxJT1qm4FjAPcDwv7wsYfV6a3uqJgwQ9jmDNDMCJTIZ3UlhJ7SMLASvzbg0yRbR3kTZfC8ScWDBWk6Ttsz0xL1K1ijvjN5kXOilPTIJH8ajbktGEWZETAcY/ju2i8KBoafQVqXUNhFdWBakK3s9KJVnNVXqzyw/n5BIVikoFwStNk1q7cdnlq4ioKuQmG0Ft9alzc+D/k0W601fb/rZaoJIhcV2Y3Zg63jevSH0rJAdItoPvsAPmkIvG+Tx55StPktdVlmxb3rvAjEcjlL80WhRpftUPIEyA9q0ujXBXCgFi9yiQ5eeZEBCHvhY/mpjrJMefjO9X/9Uu+OvDEhe3wVrAuYqcjTWC/Z+BkMmYfyNeUNBOxMwKoXC50Hqz4Otqjs954qZHj/Zn762mmxuDXUcvbPyUhz1J3agEGayThF8hch++DE4ARQvE0dOydt6SvIiwfXVL9AGWO0ggLWAYnmbmS1ahgndhXJURyARsN1Z+jaVf/WZe9QlnFRYi/nFCuNM6oXam/X1SZ9nUiVUQqJYFEhAIKQRCXDZRUlENXQh2yiDGuEO5DfyUWNO+CAChOBCRF023JS6QblCXgCTSKBdShdGRLQmkTC5xHVeWqrO3OUDKHhqZwAdFeLXcno1gy9OxAeLV2gQQanwqLqBiBIlKGn33JDQI8Da19AxA+HM3l9l6g3N5zk1lHkKZYajAn99Syw/T6Zkf30Q/LhCXoshsDcSBiYgzkaIhaGMTPbBb1dD4fxYjyA72Gnit16TNgS7DV3pML2rhcckRXtm0FAHNl2n3iTt0O5rtQ7LYn+fqnFFG/CvX58ULhbryNVrtkF/WsC+mmJf1oYym/lBfjmRDzuptKtfVAnFZGVvECI7WAwOZVKCLsGfTfCGiOIcXPTCL73Uu8o1P/97fbCsLnGkyA9fpib2NnGp9bKeyvRoUf/j6Qe+5kBIgGPkNYco0I4SAmeWgAOVmkLl6voq5ffhPqII1CPrpXYeMFKPCbpUsiRELCHMCRBTxCGHj3Co3Wywa72VMW0+RBDVIoPtkJmZE22Nz27nR1/dXR0creH3Qe8kU6QxjIyyklNb+MSzaWxm6VqN0r+88cSAG8yVmx8YilagfrzfM6uxRpiJSvybMty8aBc1qz1CV0WEt1LPJagglJdodcxtaSg7ylQ8YVV2DUX4TcA+wjh9vkcNms908F6KtL540qpCAbaDqryS0dJHbh8brH1+12+ET+a8jmFV8PWHgrM5IxKecopUp2a9j7YQxBJXhYyfdxjJ5r2WE3AvFK2PL7VnYMbc6mTyaBaY3o2qBb3CCi1MGXlT9yE0W7QUL6rOyBRVOKEtF4SwnD1fONkCNxQ7ydB17Dyv2tnoW3FzzJM12AiXYfciCpdpNpPDNZ4ANs9K4L03IzyBEmaEBqTIgOChpzaKc366bmIWbRWJSIAkEQdyiu5dJSHR3WUqlmvlR7iKzu/Ua292UB9SsR6Evx2nijeVnhW4A6zMvArOmXz2ddt+mRjz30rfwLbYimA687c0NAc7a2Vas1Sy3+HFCbE+7H4PnXZtbL2iqg6cnhr/ZNG6qyz43WU5i6W8+2TBzqNrVCS+hOJ4LRaNmswnyt64yZCxlVKFfeQ86SIOfeskXj8yiM9xHXsM0/IsLelCLFwd3CEThp9U/hJLKe/Z4hK9DGhiWlhwHNUqRrGOQcHapOZmbip1wi5hM+ARxGeJHxNpeAiBcmGFXLVPfbZN8faeeGHEhSriD794QySODtS7qFaSDj2KQSOCtrHYRICY8bKGTTI+DNT0KBG1bsra1Cj7mFwgilkjJykYKgbmgUVDLQ6Q0quz+CQl5wAq0FCqMzaEvZkzHg53TkMdFMHMVSz6g5jdB9TJaC3tyd/oSMZkiqdnwxK5d//cHr9ms4geqcnuAGfAgd/L7bSPu/X1nHUc/2tKZNz/WqC13Q6PaXQBvG7LujDJYBKv4nc/uqJzlm+7T5h2UQZngojYy2OERgLrG+SAB2LR22a3ZDnFq/gtnjQnTnG3b1VrYPqNNQk/e8PM5R+QLCWa+2QvOCHj8/oyZEX1L0rTIIo3J1NkcfTa3qW98NOh3NscspPZ9nPyOQ+rURhf2/wSZVEvHQv7Uvs8qr+eIrIjcHgrgxLao1NXXUiX0HQUet46vOidhB+hoelFDRQa1kemJ/18RE8ddTThfWgoItFUJ3zW5m1/ebWXggqDE3fLUYHzWdgvsHhOzwMPe6toyNAT1UtYDDUpb3jTayHlEPjpE7PP8U60GUdGtTuWbFGM8OuPpm2ZxUTu1mGBUXj24xt7ll4i2HA7yp3+mRVZJxS+MGW+wkJ/eMhMgY6cPwK1FSB2uPpjhaFCfK8wa9NATsk0lZesltU+10caQOk4YdV4vlzMJpzRt/t6jMA8y0tgfGcFdQnyrpgRV/N2k78/xjcUfiYqZ2rE8j/HkD8X+DDfajhSYzY0+9dJEFdd5r/PGR5AFh2uPJ5sHIUVRpQ+Qdhyt8DKeGtWRbVGDhHi+dclxhU7ektnbPPHyFRJ38AU/wHUEsDBBQAAAAIAAAAR10XuIfs2gIAAKMFAAALAAAAd3JpdGVyLnlhbWytVMtu3EYQvBPQP3Sgiw2IK8kWdFggATaRkiC25UCbIEeil2xyB54HPd3UWj7lI/KF/hLXkIYkH3ITL2QPe2qqq6vnmH58zqc6Jh4k2uqeg6cv//5HNzJl9tt/rtd0m5LRpvymX1Ls3YDs3yQEpgu6kjvxaZT8kBBGMWcuRdpOu+BU8YkNz033r5Q8ddJ6zlxOU+qT9+lAthdKeeDoPoPV75vbm+vttrm93ly9u16F7oSycEevzl5d1udn9dnrVQH7bk+LGpzHR+f6XjKQcwoz7jSqYXugzdUb+mP7/oa03Uvg1bMXWEUOsqY4d6HRgzRzf6pOtM1uLBWv6aeKaDNZiimkSUlTbwfOQhIHF0Wyi8PSVpBno8h3bmATnSv08gnvTnasoidA6hwPMSGg3TR8Kxrtm4SeHKonxLGjLnNvSsFFF9iTcR7EpKORDYLoCnA/T84bwQZFuMVN77cLHdcS53bvTFqbwBe7TDJa+CLtVPKd1KPnWHNr9R2q6O/rLGXl5aqqAij7NQ3Ff/VF/fp8VzurP7LVhws+v6xbK17ueATimgK72PiUmeiY/o6oOxQ5QB7tfZtuN9RDqdqmWLR68eceYtDlywoY23s1gQIRLZ/axWI+cYcqZ2nkU+GM4gPnD106REB5qZ7kr+kHF1s/dagQO0bTU51BYcNqnqEoi3mhQEa/AajVsKxK06ZoIFvemLknYMsCwBhdBO95aGfKCLzUS8tLX/FXVsuoFD8pFXeAH/IUlS2z0veudahjzzmKavHyr5N/iOeiizpZipqK98fJZciwkz7lMhQlY1UZjtE1Ol9TnmJTtIZVlhgj18zylEg6Z4/RIcMHjyF81KixTTqHWu4Qa2ZfLQtSrNOoCw6j3xQHP+wrQRPFDftdyvr9MoCGzOO+Oqr0g/NgerTgzcGppim3UrswpmxHVdU55XKdNJY5Km6BxhJYZCi7JjRY/icDU6ffEr4CUEsBAhQAFAAAAAgAAABHXbo4FLOQAAAA0AAAAAoAAAAAAAAAAAAAAKSBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAAEddtNr/QmoAAACAAAAAHQAAAAAAAAAAAAAApIG4AAAAY29uZmlncy9wbGFubmVyX3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAAEddmFcxJlQBAADvAgAAFQAAAAAAAAAAAAAApIFdAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAABHXen1ArBMAAAAXwAAABAAAAAAAAAAAAAAAKSB5AIAAGV2YWxfY29uZmlnLnlhbWxQSwECFAAUAAAACAAAAEddk6/TBMYAAAA7AQAADAAAAAAAAAAAAAAApIFeAwAAcGxhbm5lci55YW1sUEsBAhQAFAAAAAgAAABHXZt7Z8ReAQAARgIAABIAAAAAAAAAAAAAAKSBTgQAAHByb21wdHMvcGxhbm5lci5tZFBLAQIUABQAAAAIAAAAR12EnRvPUQsAAF0aAAARAAAAAAAAAAAAAACkgdwFAABwcm9tcHRzL3N5c3RlbS5tZFBLAQIUABQAAAAIAAAAR11Kr/xcWgUAADQLAAAdAAAAAAAAAAAAAACkgVwRAABza2lsbHMvc291cmNlLWltcG9ydC9TS0lMTC5tZFBLAQIUABQAAAAIAAAAR13GfNkbpw4AALQvAAAzAAAAAAAAAAAAAACkgfEWAABza2lsbHMvc291cmNlLWltcG9ydC9zY3JpcHRzL3NvdXJjZV9pbXBvcnRfZ3VhcmQucHlQSwECFAAUAAAACAAAAEddF7iH7NoCAACjBQAACwAAAAAAAAAAAAAApIHpJQAAd3JpdGVyLnlhbWxQSwUGAAAAAAoACgCiAgAA7CgAAAAA", "sha256": "ac1f7acd322707b6ec735fcbb3e3caaf5c2b5551a686d57f9b9b755ca068aa90"}, "tool_paths_v23_writer_thinking1024_budget3_v1": {"base64": "UEsDBBQAAAAIAAAASF26OBSzkAAAANAAAAAKAAAAYWdlbnQueWFtbGWNwQ2DMAxF75nCC5QBcmOGDhAZMMWSsdM4AWX7QqWeevvvv8NT3CmCUisoyU9KmTMJKwV8kdY0C7pHeNK7Xcgo432HhXwunCubRhghFzvYL0ABdm8EW89WN3J2WE3ETlpg6qBY+SAolM25Wun3RC5D8Dalb9JjAHjAbLryK2WsW4QsqEpl6LjLvz0L15/8AFBLAwQUAAAACAAAAEhdtNr/QmoAAACAAAAAHQAAAGNvbmZpZ3MvcGxhbm5lcl9zYW1wbGluZy55YW1sPcpBCoMwEEbhfU4xJyhRVOhcJlgdk5CaCek/0OPXdtHdB+9BziZ9hXVh8rfBQVtoX97nnwvT5N25voMamiFAi9QX0+DHySHlWnKNYdN65MiOKNftabsEJLWYcJ3oJlf4vw/bo4BpnBf3AVBLAwQUAAAACAAAAEhdi1Wy0FYBAADvAgAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbK1Sy47UMBC85ytKynmHmWGB3SDOe0ELEuIcdZyepDXxQ+72MsOJj+AL+RKcBUZiz1OS5YPtcj26xYdromnhYjjIpK+UfFokTJsz+QW/fvzEAwfOZBIDPlMmz8ZZ64trK/garAQeoUbZqgIo27rrBo8RiczNN1qcY1WITzk+sedgmEkxMAd4Ji2Zx83VxTXGPq0hVPoO280OF7T48jcxZPYkQRFiGNeQvARRE4dvYnMsBsJBTqtD5rGxmPq0kt2/wf9o8VjcwkXxr4znHo7MCT6qIR5Q7Q80yCJ2hifVZ7pjh9stXqLFR/FieIqOhrJQPkNTZhpxiLkuV7Rqmi4tN55OfZWbivUWjxy0w93ufr8yfcoTBfnOGVORkcGnVSG/h0WjZR0i45Ndkni9f/f2rrFZwrG66P/MWNcAEtxSRu7XXKbZ6g+WC9eDy92hjBNbh912f9v8BlBLAwQUAAAACAAAAEhd6fUCsEwAAABfAAAAEAAAAGV2YWxfY29uZmlnLnlhbWxLLUvMKU0syczPs+JSUCjJzE3NLy2JL05Nzs9LKbZSMDYwAArnJlbEl+Tn58QnJ+bkAEVN4YJA9fG5mXmlJakgxTDR0qI8INcQqBcAUEsDBBQAAAAIAAAASF2Tr9MExgAAADsBAAAMAAAAcGxhbm5lci55YW1sdU9LakMxDNz7FO4BnBIasvCu+96gFKP6KS+msuRaMuHdvg4JIZuuhDSj+TBUjL6oDkznrYmdUYs6WJEtZQLV6D+ovl93V2VBin7FWiEcwtv+OxQLv2DhcoD9MWRzhdX6yFaEo38pnGks6FuX2kxfGwEz9l1d3B1KWdim9rRhYXQyrA1LP7hF37FB6c+xTIQm8/PLzTzYwR7/13kq65Pn7fDwTAq1UeF1t0EltxQFIrkk68B6mrhJatCnVPSzAf7DQOx6J/wBUEsDBBQAAAAIAAAASF2be2fEXgEAAEYCAAASAAAAcHJvbXB0cy9wbGFubmVyLm1kZZLBasQwDETv/gp9QAjtpYeeS2GvpZeeihMru6K2FSQlu6H03ytnt7DQmzHyzOiNX1kKRNATi3UwC6+kxDVmEJwjCZy2me2ESgqTcAE/A6kuCANmPvfwwQuc4opQORhzVog1/d0YUNUZR8MEyouM2MMb2iIV5hypguHFYKkJBR6fHuDMkhRGXlGoHp9Dc8PLTWBAVyWWbg+R6QvzBlTmjAWrRfPcUEgtfmEH0aCwGtiZXSGOFij5FE2E8m8V47+c3Z6eKwIPirLGIaOjOApq4+LRaqLm1IfDdKeQGPW679Vla1C3MnDuIKGOQgPu44IZ11jNDzpzVRook219eOEbr9UFYKKMNRbUzgcnFKwjOsG8NG+/NNRdYsmmwOJuIzcSjdNElz687157hfHYFM+UMzSuns0fJNIxSvJM3uxdyfHotbi2E1v8F9xKC+HQ1nwO3/5FHEn5vC41tzg/4RdQSwMEFAAAAAgAAABIXYSdG89RCwAAXRoAABEAAABwcm9tcHRzL3N5c3RlbS5tZIVZTY/kthW8C+j/wFtsQN2L5DhzWq/HiJEPLHYdGz4N2BLVokcSZZLqmUaQ/56qR0pib2zntDuSSL6PevXqsX92i9LeqH+axevh809PJ/XJBDdcjYq9UWGZ58GaVgXXxVd+aENYjDrflGlttNNF2RiUN7MLNjp/O1R6alWjh4HvwnIebXyedWz6k/pXMMpNw012nr272hY7R+eGcFIfvQnG49h5OQ+2UY2botdNDGlHO86DGc0Uk1kjDjAh4is+j0Z19k2MdT5iT5gX3OIbo7g24stwOlSH6nPUPvJt05vmhRZeTHwOUccl1IUXan0EK38xTVR2CtEvTbRuwlNuel7s0B6q0UTd6qhP6ulN7BX7zNuMVTTE9Ppqna9Vp62EZH/CXXCkd23a+FDBZwYVZ6jOu1G2knif1HcWX5s3HoCoTdF21vigXm3slb+ory7YSdlOLZO+4iR9HszXcPmT0a1s481grhrh2wKpeVAtqcJO9R4pmNg5ZDonGKnRIeA1jrdBUt4tA7L466IHmoEoTK41atIjQONUMNo3/XOwI+zwzw3ePQIlMEFwFRBcbN8qWWTGs2lbbBpqbhMBxrNFIJGDCSkAJo+Dni6LvhgcaLw1IQHp4vXcq8nYS392jERvJvH0vFxU410IBtAZEVwA5VFdsbS75VWwZBngqL5oJlY1i/eEVmcHbv8PxJxuOk8cDOtZTPOhah08oKGzN1cuch7m096MOOR+cF6ie1Lvr862ashxo3mvvRtMATXB5RPOAepDr/QGlBIaybvO6GCR2JP6PkEgVZJ3sKbRS8ho782hev/x+w1peKKjGpfAFIzwWKoGBspWgpAErDUM0bzFEgTPjAvsZNhfvY1GHii8h7m/YF9BAXLxKhFU8rw1gz0bxIHOyiqYPQXWhVjd2q47VJ0bBvdKd3kQYv+tk+DCUEcKIh5r9Wr0C/wHDI3PJdhr3x4bQdBaa3oKr0SyZJ3BAeCHRRKB+u28TiW8eFPTRJuN6W2LaCp8a9v0sdQz0/KtDQ3M8GIwIgTAi0nKL9OEx1KkK0WgeDt7WdbMf1oSHDvXIDOoEW9QowAWDmCaJlQJiChVXGd9iCf1jXeatgg7hcSVqOnz0oKmZBXsmU2OObgWB4xY5ha8GhyiiMyO+C4TQ7SjcUtM6Nl4oeBaVHTMXPegWpcqEAExgkJEimTQk3o02LVpYD9qX0W8fNcTOLOdkecJBU748S8uZHgbPTPUYuNqVj5KSsQ2Ng43JDzVxiLgB8dMV+vdRIJ6F0xc5tWSFGxdxhGLzWwmRKwBLUgbQ1Gyoslp7VoNKuA/Z/cGKlWu62guYpbgDUTI8Yfq4y32bg19jWhm4kxVfQRtorWo8GKHITE38svWB2ryLVLVpIKQ6stQyYSQl5Iec7lxnfP2YhMUCp8zNaFUULH6VrQETcrNvu1d9NX5lzDrxmwwzB4q2RsuuwElCPiBq648HiQKAB+PmoV3zN6BDXo2YpzbkaqFLSWm05asokIOVWkyjYJXbJaSjtwBySgeGfSTYZ/7dbEg/Wfi/TnD4RkL0LcBEaeefnz69LMqk/CYaAKAQkByEMPKYMvU9GgLpj1BubBCW/c6MWeHCknInQaLd3iAi9U9oGzicc2OQVjtdPs/1btTz8qLzJvuYuaGlfqRqkMF8zpY0K/eYF8WjhE5IYW3yqYUpH1vAcNekif1Qy/NpzXiE9tm4HYxEZ5QkfQsrhSs7AykvrJTMyxsrWq+ZXScohsTfPkt15/m29ePop4KP9bOIST4gckodFJ2u2hzdX6k1Wj0hAPJEYxFXYRpRw92LXZjJ2RiY498RB1e/gSBkHkyEV/Nngl2IZtJR+bnaVNJKPZjnEjUrA02NxNt3Ddgpv/8F9XDuaD2qCQ0aNEKiit5esh9LmmmLCdXe2jBI6CWcANhBsHzm67g8eDOoIBkgAixxLjcgiJp7T7CiqKSt64oRYjC0zOyphvKldmwiFHzxC47XEK/LOlvs8O/AU4uYVUZ7MWG+hq8tKqQdmEhsyGXEoUp/g41I+0DwoswcZRZtXox0JQg/p701WVSk9qcnaXyWWJgr1hjtU4AX5EIgW0qTfUujvO7SaaLZ/Tn5339aWy/fijUS53rgYh+F27j2Q2wqtGp44NzqBDAii2+vO9zRc3UpAaUE0gi3urcaSFn9FrY5uhJwQnpoGrBUQqncEThHyKqQdtXe0kVpS1IjqmXwtvCywjmdtLZCX0MSai3BF9gGrMKypWtxaTt4V+f3n9br2onsboh02mZpMAKDHtQN0xpjRcQJJ+o2u+mK2Iqt9xSrci7egc9OjyQkcs8y937bYwfIWfhYSKEVbFkqmONgtEf2CREUZ4RQ1jn6TrVEYa5HOuNFWBpUfwEd6G1rFRknuLqtRQoHzszidpNPkiiLTpPIYZ005DszJDmNWpriy+zevJ6WptDM2g7Zqk1k25b2Q74PVSFrMHf8xIJAhSeqGkKSjjg0E833b+ZmKZaZv/TPjiuY6sYKDE5JvtbaEk7szU/HKqfRJDJ3INhampSg7dErC008jZwsc136Nd4e90qAiY1CzuwEe5PBVGrNPJQbnAlzcgTNb70N0H2NuI1vXPCGfpOvWN6gF9ZiuNIdvlCTHwxQ4L1PXNwNnkISeP5F7OtKJWT+jGl/kvSwCkRuk5vbXBeJerdOJ1UwLsPf/9eup1EGl2SlVFOvhi4JiBAh5IMkkQloWzKbC3h1D4nwQokC9jvSuM6+MW3cHtJUzlik8eF/U7hg9Ql8wtn5PS7yS003s4xZDy8SYFCxFNcQbVL11wmC1UmTIn5EjKJDAwolPzK8EVvduVaVu092+RudDwKcYM4Cwriu/qefjZCGsIxiYnjkQ3F8z/mjcRhuG5qIXRRHE2DcSNydAPJI2K3zJ5Jby+SlBf8RU1Jvt2pTXiiiJOcRvZjgg7VPkkW4eOIV2+Ce72SSm25xIwUPUv3HnWZacw6TInoMr6xobic2Vo6pkSyHGRAkpCS9TR42JgrXbdH2argMPHjIV+2bDWb6GbVWrsuY9y20Tw7xmHy/orpTG4ajE4VMO6Cl8I+hYH5jynQvF37uLBIb1LIrdWXycGORokcXicm0J3AUaaDEndqg93vdPWsTU2zAOukv3R1ksa78uJMKgssf3dfVyR0c30dCXgdt110PfKCRXKxKpx93A71plpX6Mgl3Bfx5RDGT/bbAhkwRv1Cako9mE3g/1TSH0D7UK3YFkmuvsCDVFG6z5DGTcUC8eWTNrO7vOWm6V4qhbGEfRIkedwpJvaitxeqB1OfMnrtfUBq9QMbMc1MECyqblMVDyKb9wlSzJbbWEWJJJc89X4BJPsU1z9uTrc7sHbPv0Qm39CtlAXBhbZhr9t4kcxKNxQQrma9As4NJ3CGFFDKh0zebxmZDgE01nFSJFAi49/DNs+sxhdqrJpng557I60qfZNkiwL5GTRXuSbKQxwb70sC6/2RVfOH/J8ZYBOVIqv83V2YzDar6qsSBad73eIg9LJXWHpSn0kJ6ep42eaUPGXvjjJypLWqmFT0foq8T1S1awH6kSbTU1VxBNmDjSM4F8h9APjh6Lrjml3Zqt7K1d1ZnW9EqsCr4d/A4JZzaRL6vnPyEqkQ4xP25l4F2bBjyI0zDA/drbT4pP7G6aXYLhVkZs8CsRJdYKAwb6fPsNGhpdAQ/Yvd04WJp87S8gMJ0+rdcqGmlFpmEyl0Nw6Um8TbKophj/eYsKvqI38ICSIAEwem/lKMddtvAXJNyJK/mIfq3/BBW/+8f/ifSgy0kqZlypc4bRqZ5R5Axi6VJvqdxHNdPoJIvZdfO/KV4aaPEBuNHmLylYSk835IV+IewARbkoXhVP0XUEsDBBQAAAAIAAAASF1Kr/xcWgUAADQLAAAdAAAAc2tpbGxzL3NvdXJjZS1pbXBvcnQvU0tJTEwubWRdVk2P2zYUvPNXEHvpxdIe2l4S9LBB0iYFkl2k27RBURi09GS/rkSq/PDa/z7zSNHW5mLYJvk+5s0M2TSNsmaiVzq45DtqeJqdj6qn0HmeIzv7Sn8hz8NZx4OJ+uEcD87q7kDdU9BlN1ZIU8+Rev3s/FOYTUdLQG1srztnsYpYZhzPmm2I+FLizd79R13UbhhGtoRFzRGBgxuNxAs4v3MnTfbI3tmJbGxVg6rVGxqcJx19CpHtXptaW6QQN7rn0Lkj+RKu1Cmd6sG7KVdsupjMWCtQq3p3icdeTxRNb6Jp9Z+BtE92G554HLcFGf3M8aDLPznuCwA3auCRtrORPXl/uC0btmXDdp+M79v5nBMavw/6n5ummVyfRrrZ6JsPHx/uPz9uP919fHezUVgiG5LPS00D+Nxzs2SUJDf/tvrBhIBomk5cEFnwKDE32roIkIBL9LxLMo4MyEbnKp1X4UAYS+emCTW1+hEY4Z8ZGFYwQwbuN46rOUv9KVBZCoJEyavYRvKzJ3xqE/IcrIl8pOtQT9Sl6HyrP5Ppc4Df/7j/pD2FNMYcGc1ElNRTq9SHWHknOwuOoIggLbwxGC2Fg17lbfXd6BH6jBDeC88KZqocDjK/uRLSCCStvkdw/8wYOTKj+UkKdxbENXpPNglLhexml9Pmo8oNqInDFZbNhdGZJ01jXcO2p1P5mhnWFJbLIPKfPc1hI12rnE5aslclgP+ZrysllNgCRmHsznRPJHRamjZHw6PU2X6HnfO8l/B7w3YDuCcMd2nS0jM+0Sj5nDbZ6CUsRrHfo4XBbdSi6YH9VOLVLT0PAz76zLXuYOyeWo3UlkSJvXu2ozN90OgUdZLtmMJGgaA88MIgOpoxGZACCPolRsiaDvIHOE5eIAsIvJobKsYGExEsRP3TzzqQuE54nYOC0/MIQlwIKLLVncl0h66mJTlggy85N95GnkjtUo+wrX7rynSdBd8tGV8wJ2BoIbQiLY+eBDEhypA5jxpimvWAIUC5IPCdBZwhdR2FMKSx8pyLbFdzXY5UzR7FgBn47uhgjgyp1pjZmqSYqlCxzKuRrSUYHdB0DvuNHlE6OkS5enSAQQm+VfmvMajVAC8WnSe+T77gjVkcuC/JL9WgweUCWbpEKEArwRDlKKkBk2Evx5ES3ORQGDx7Pko9e2968j+El45/lxEV2WW8YcZrCx9cBwfqr+RY2wESSyWt/hU5vZm5VxIHSsMZPWev+lE3b3TzN351BjLZArKBT7/cxmmutp3Vs83LQHjtswotucCg7FkCgFZWGr6gs8yjyuEBVZE/UoW7WBywiCmIyQ2ovEKefdUg2kEustUwRQmFTPPigqM5u4R7D440y1GOKh68S/tDJb3wOmRaSMHLtT0y/G1xvrdl7PV+filSqTq+7CVcs6FZ7NTQLveFH3JzZgkHibKoK4++qEpFf5ZLCrSMyRers4M3uJ0SpuovrJK+nWCC6eZ7vg4ZYP11oEKEarbVASGpict+V9S6Ro8LJau1bmSDotM8cse1x6bcivkJpCdTni1ySx3La8hkT2qyZz58fXx//+nh7vH9injqQjwY4EUGFVxjQbylPFCtaJgGk/2gvFZeCAC0uIK7XS1ty10R5CEEwzDzPJ6LkZ6gDmAu6FKP+f2fGPawlbK3C/cWossQ3n159/lrfTJgUh6vhi43XxizR09Bftb3gboHc/I7g6WtpbrFSLParRG/pCMLiSj7dSbLqhGcQw9QzkYJOrlu40eW26Kg0ax9cXlcQfC+PB+zKi/MnVIUC6kvupeGhR5xmwdotpj5Oq60WE1vNDx950ZasEPVuQ31DVBLAwQUAAAACAAAAEhdxnzZG6cOAAC0LwAAMwAAAHNraWxscy9zb3VyY2UtaW1wb3J0L3NjcmlwdHMvc291cmNlX2ltcG9ydF9ndWFyZC5wedVa/3OjNhb/3X+F6v4AbDFNOt3eTTrMTTbrdjN3l2Syaa+9NKMhINvsYqAInLg7+7/fe08SCIyd7LS9m+MHG4T0JL2vn/fEdDp91SRLUYvEZ8VikaW5YLJoqliwsio2Io9yuI1XIn7PxKOIG+jJ7rcsylm0FHnN5Ps0y4LJ5GaVSibrKE+irAAiizQTrJFCsqJK0jyqtkw290AzFhJb01ymiWD1CuZryjJLga6E0ffFYzA5r1kuNqJi6bosqhppwBqifAkDYWaxibImqovKZxGrxEJUAldZRnW88rFvLWTNIpinqtMil8FkOp1OFC0WVcsyqqSYLKpizZKoFnW6Fnqm9tln+Psb7MSMW0VylaX35lH9QUPQ1GlmWt/JIjf3hVRTwLJwoJnhCh5Nl6qlLlc2mY5Tbcu2va3FukTuts+w0MlkkogFW6a1WxVF7bMXsE3pnUwYXJWomyq3iAZVk7u3DvR2fObMzuBX1hWN9PTQO5/FUQnjBC+aumzq8KZqhE/0xi9cBnQNj498pS/hd1EmhaeXlqQyBn2q+ENRvZdlFAtXPILY47QOL4DNeq3pgplmkHTCXFyYafHYZyFzvmwpOChr5GfXI0glT9LK9TQ9vGIglKJgJQvZbb//HfUSsNBD/YP4IXGBNTTWXoC3jyPU00gqAAPDe7WwANQPLedL5nSE7kZmD8RjLfLEJVqlxxaw2xIMBzQrEPkmrYocKbvO1c83by4vrk5v3qA8HS+QsLvahW6oe1LAWOBr6SltACnjxiSMVC1It50V6XdL6JgCBPKi7l61jD7pcSAu8jrNG9E2VkI2WQ3zoW62o2GZldjMyBBJB2dyVTzM6qLMwO4zx7PnVSQCpcZxAV4jDNlRf17aVRAlmlt6iKwTUMkgETjKBb7UVVqiCOB9kW0EalcaK932PKOAmcjJFiTp23E3URWlUrAfwfmIeVUVleucRTlyBTwZbHuxZej5vgfVbQX7LZi/lGw2s3TGNkq17rKAVWlDUd6XZ0UcofPS9rwukiazjARnrUSwaLJsjW7PrZzb09m/o9lv/O6Xhxfu305+CewG74UzIDK+ofljKWL08ehmH1MJwlyyq229KnLjwPJobTYB8iNdUoS12jmBY/SqAQsGE7pr1SyLtiAQ1DG9L/xFS5BV7FgLA0a9h/gCg9WI4F2R5qjM7guas9OPHeV1zVigynmapzXnQbmF7esXwUNar7hsFov0ERYLr7yBEgN/e3qONjxUdK3bqEW4x67/mGrtjGwlqAjgJJXIQOAbwetCeeLd+cZF9laFa7ldQ/R+D9obbcBzDfXNvkgwQVSW6F3MGnrqTz2eVv9zpRHrRg5sQEMIAgEgkwVsDX01SFkLtG8FNN3t0Z02gRj2kHNwcYbpcAs8ToCfbuf7PPMqaErkvKvc4OvLi5t/XZ/fzF/9fDM/u3w9D51jEL56eXH5w9v59Vt4S61DzlydX/GLS35+8Xr+kx4GLa/P356++sec4/2POPrygp+9mZ/9Hbv09gGL0VtIc8BOgFC2Pfv1MbiRffnYl2NArIBxdgT8nJ2yBQhlBToFcRRgmABFxT+kCFAoSxgobUzMlYwgxipN0E6rqEwTw3uRpOBZiKTymRAZnCF48RGw+Agu6moL86/DDteoO65W7kKfAIDB5vb4zvsWZJku0zyE6BPVEKDX/pSTlXA+9WkvE+21ww9TTUc2cSxEIpLpCUGJqaIxPVH//hR4IcHhTU8w5Pco6zdIfDr1vI8T8RiLsmZz+oM3gPUYBvADcxIY8acCFZfX21JMT/DXFV7AObo0zvVbtQDh3Z68PDq6+zjRsUQrmQLB0X0mQmRJ9+grQVGruvXvIym41Ww9e5OySvPanb69/OH6bM7P/3l1eX0Dmvfj/OLm8vrncPoFiiZImnUpdTiDCAUiJImiC2uBYHAjcLsAsV+nFWgXap2e1dHRZNlEVcJJobjjIbvo3nK4gAwRFAww4mCLEKhfUbiW9PsT/oIqElkzI/tC0T6EFelyZjGMR9005oGg8yEJlcWAfYS2jRA07D1L8qGobgTgbLfx1ORj2LZDsC8HCBZHYGZEeJBSJIpleAPODRk2QBqkRzIEjFNm5IJVaCQa7sAfo8eFdqCAgQ0FCy59j1I4Gq8imsNZLUgErvoIXSx6b5rnae/9HXkZ0EJRgfDqNt3qnBdbRKBfSevkNJQjzcyKKJFqKvDcJvaHINJjD50EDVG2DUO0ERFcVY09orfaM0luHCRXmuvcweD7oshcTQqTAoJ46rmDch6iQjN6QBt6REgTtrUhiuaNUlpo+qxrs6xUw3KD1aiDdvD3ACYgfPJoAyxC47CDtn6JPhdgdlPWsAEw/ybNEr4Gz6aMGO/Ap0S4Hg2Dyi3I9R1sIaiLdeYYYZueY2DEZIHQH9PTVsOLfJEugbR+oeXVUqpElPAaMgxXoAJB+Aidpl7M/up4nYLSihHdEDElPWqbgWMA9wPC/vCxh9Xpre6omDBD2OYM0MwIlMhndSWEntIwsBK/NuDTJFtHeRNl8LxJxYMFaTpO2zPTEvUrWKO+M3mRc6KU9MgkfxqNuS0YRZkRMBxj+O7aLwoGhp9BWpdQ2EV1YFqQrez0olWc1VerPLD+fkEhWKSgXBK02TWrtx2eWriKgq5CYbQW31qXNz4P+TRbrTV9v+tlqgkiFxXZjdmDreN69IfSskB0i2g++wA+aQi8b5PHnlK0+S11WWbFveu8CMRyOUvzRaFGl+1Q8gTID2rS6NcFcKAWL3KJDl55kQEIe+Fj+amOskx5+M71f/1S7468MSF7fBWsC5ipyNNYL9n4GQyZh/I15Q0E7EzAqhcLnQerPg62qOz3nipkeP9mfvraabG4NdRy9s/JSHPUndqAQZrJOEXyFyH74MTgBFC8TR07J23pK8iLB9dUv0AZY7SCAtYBieZuZLVqGCd2FclRHIBGw3Vn6NpV/9Zl71CWcVFiL+cUK40zqhdqb9fVJn2dSJVRColgUSEAgpBEJcNlFSUQ1dCHbKIMa4Q7kN/JRY074IAKE4EJEXTbclLpBuUJeAJNIoF1KF0ZEtCaRMLnEdV5aqs7c5QMoeGpnAB0V4tdyejWDL07EB4tXaBBBqfCouoGIEiUoaffckNAjwNrX0DED4czeX2XqDc3nOTWUeQplhqMCf31LLD9PpmR/fRD8uEJeiyGwNxIGJiDORoiFoYxM9sFvV0Ph/FiPIDvYaeK3XpM2BLsNXekwvauFxyRFe2bQUAc2XafeJO3Q7mu1Dstif5+qcUUb8K9fnxQuFuvI1Wu2QX9awL6aYl/WhjKb+UF+OZEPO6m0q19UCcVkZW8QIjtYDA5lUoIuwZ9N8IaI4hxc9MIvvdS7yjU//3t9sKwucaTID1+mJvY2can1sp7K9GhR/+PpB77mQEiAY+Q1hyjQjhICZ5aAA5WaQuXq+irl9+E+ogjUI+uldh4wUo8JulSyJEQsIcwJEFPEIYePcKjdbLBrvZUxbT5EENUig+2QmZkTbY3PbudHX91dHRyt4fdB7yRTpDGMjLKSU1v4xLNpbGbpWo3Sv7zxxIAbzJWbHxiKVqB+vN8zq7FGmIlK/Jsy3LxoFzWrPUJXRYS3Us8lqCCUl2h1zG1pKDvKVDxhVXYNRfhNwD7COH2+Rw2az3TwXoq0vnjSqkIBtoOqvJLR0kduHxusfX7Xb4RP5ryOYVXw9YeCszkjEp5yilSnZr2PthDEEleFjJ93GMnmvZYTcC8UrY8vtWdgxtzqZPJoFpjejaoFvcIKLUwZeVP3ITRbtBQvqs7IFFU4oS0XhLCcPV842QI3FDvJ0HXsPK/a2ehbcXPMkzXYCJdh9yIKl2k2k8M1ngA2z0rgvTcjPIESZoQGpMiA4KGnNopzfrpuYhZtFYlIgCQRB3KK7l0lIdHdZSqWa+VHuIrO79Rrb3ZQH1KxHoS/HaeKN5WeFbgDrMy8Cs6ZfPZ1236ZGPPfSt/AttiKYDrztzQ0BztrZVqzVLLf4cUJsT7sfg+ddm1svaKqDpyeGv9k0bqrLPjdZTmLpbz7ZMHOo2tUJL6E4ngtFo2azCfK3rjJkLGVUoV95DzpIg596yRePzKIz3EdewzT8iwt6UIsXB3cIROGn1T+Eksp79niEr0MaGJaWHAc1SpGsY5Bwdqk5mZuKnXCLmEz4BHEZ4kfE2l4CIFyYYVctU99tk3x9p54YcSFKuIPv3hDJI4O1LuoVpIOPYpBI4K2sdhEgJjxsoZNMj4M1PQoEbVuytrUKPuYXCCKWSMnKRgqBuaBRUMtDpDSq7P4JCXnACrQUKozNoS9mTMeDndOQx0UwcxVLPqDmN0H1MloLe3J3+hIxmSKp2fDErl3/9wev2aziB6pye4AZ8CB38vttI+79fWcdRz/a0pk3P9aoLXdDo9pdAG8bsu6MMlgEq/idz+6onOWb7tPmHZRBmeCiNjLY4RGAusb5IAHYtHbZrdkOcWr+C2eNCdOcbdvVWtg+o01CT97w8zlH5AsJZr7ZC84IePz+jJkRfUvStMgijcnU2Rx9Nrepb3w06Hc2xyyk9n2c/I5D6tRGF/b/BJlUS8dC/tS+zyqv54isiNweCuDEtqjU1ddSJfQdBR63jq86J2EH6Gh6UUNFBrWR6Yn/XxETx11NOF9aCgi0VQnfNbmbX95tZeCCoMTd8tRgfNZ2C+weE7PAw97q2jI0BPVS1gMNSlveNNrIeUQ+OkTs8/xTrQZR0a1O5ZsUYzw64+mbZnFRO7WYYFRePbjG3uWXiLYcDvKnf6ZFVknFL4wZb7CQn94yEyBjpw/ArUVIHa4+mOFoUJ8rzBr00BOyTSVl6yW1T7XRxpA6Thh1Xi+XMwmnNG3+3qMwDzLS2B8ZwV1CfKumBFX83aTvz/GNxR+JipnasTyP8eQPxf4MN9qOFJjNjT710kQV13mv88ZHkAWHa48nmwchRVGlD5B2HK3wMp4a1ZFtUYOEeL51yXGFTt6S2ds88fIVEnfwBT/AdQSwMEFAAAAAgAAABIXRe4h+zaAgAAowUAAAsAAAB3cml0ZXIueWFtbK1Uy27cRhC8E9A/dKCLDYgryRZ0WCABNpGSILblQJsgR6KXbHIHngc93dRaPuUj8oX+EteQhiQfchMvZA97aqqrq+eYfnzOpzomHiTa6p6Dpy///kc3MmX223+u13SbktGm/KZfUuzdgOzfJASmC7qSO/FplPyQEEYxZy5F2k674FTxiQ3PTfevlDx10nrOXE5T6pP36UC2F0p54Og+g9Xvm9ub6+22ub3eXL27XoXuhLJwR6/OXl3W52f12etVAftuT4sanMdH5/peMpBzCjPuNKphe6DN1Rv6Y/v+hrTdS+DVsxdYRQ6ypjh3odGDNHN/qk60zW4sFa/pp4poM1mKKaRJSVNvB85CEgcXRbKLw9JWkGejyHduYBOdK/TyCe9OdqyiJ0DqHA8xIaDdNHwrGu2bhJ4cqifEsaMuc29KwUUX2JNxHsSko5ENgugKcD9PzhvBBkW4xU3vtwsd1xLndu9MWpvAF7tMMlr4Iu1U8p3Uo+dYc2v1Haro7+ssZeXlqqoCKPs1DcV/9UX9+nxXO6s/stWHCz6/rFsrXu54BOKaArvY+JSZ6Jj+jqg7FDlAHu19m2431EOp2qZYtHrx5x5i0OXLChjbezWBAhEtn9rFYj5xhypnaeRT4YziA+cPXTpEQHmpnuSv6QcXWz91qBA7RtNTnUFhw2qeoSiLeaFARr8BqNWwrErTpmggW96YuSdgywLAGF0E73loZ8oIvNRLy0tf8VdWy6gUPykVd4Af8hSVLbPS9651qGPPOYpq8fKvk3+I56KLOlmKmor3x8llyLCTPuUyFCVjVRmO0TU6X1OeYlO0hlWWGCPXzPKUSDpnj9EhwwePIXzUqLFNOoda7hBrZl8tC1Ks06gLDqPfFAc/7CtBE8UN+13K+v0ygIbM4746qvSD82B6tODNwammKbdSuzCmbEdV1Tnlcp00ljkqboHGElhkKLsmNFj+JwNTp98SvgJQSwECFAAUAAAACAAAAEhdujgUs5AAAADQAAAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAASF202v9CagAAAIAAAAAdAAAAAAAAAAAAAACkgbgAAABjb25maWdzL3BsYW5uZXJfc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAASF2LVbLQVgEAAO8CAAAVAAAAAAAAAAAAAACkgV0BAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAAEhd6fUCsEwAAABfAAAAEAAAAAAAAAAAAAAApIHmAgAAZXZhbF9jb25maWcueWFtbFBLAQIUABQAAAAIAAAASF2Tr9MExgAAADsBAAAMAAAAAAAAAAAAAACkgWADAABwbGFubmVyLnlhbWxQSwECFAAUAAAACAAAAEhdm3tnxF4BAABGAgAAEgAAAAAAAAAAAAAApIFQBAAAcHJvbXB0cy9wbGFubmVyLm1kUEsBAhQAFAAAAAgAAABIXYSdG89RCwAAXRoAABEAAAAAAAAAAAAAAKSB3gUAAHByb21wdHMvc3lzdGVtLm1kUEsBAhQAFAAAAAgAAABIXUqv/FxaBQAANAsAAB0AAAAAAAAAAAAAAKSBXhEAAHNraWxscy9zb3VyY2UtaW1wb3J0L1NLSUxMLm1kUEsBAhQAFAAAAAgAAABIXcZ82RunDgAAtC8AADMAAAAAAAAAAAAAAKSB8xYAAHNraWxscy9zb3VyY2UtaW1wb3J0L3NjcmlwdHMvc291cmNlX2ltcG9ydF9ndWFyZC5weVBLAQIUABQAAAAIAAAASF0XuIfs2gIAAKMFAAALAAAAAAAAAAAAAACkgeslAAB3cml0ZXIueWFtbFBLBQYAAAAACgAKAKICAADuKAAAAAA=", "sha256": "7299ebf74d5d7c7972652a0c47af5c09d9811d16d39c0858b1a75c02cac3bb30"}}')
SOURCE_BINDING = json.loads('{"plan_sha256": "b422f6e45645b119d9781e986bfb14c1f63adc0aec75972c54b91fc7f03cb051", "problem_only_input_sha256": "711abc3a0494271c6e65c193f69f8a3f31b0e7eed0e2ed8105e6f64995c84453", "source_sha256": {"scripts/prepare_broader_scoring_comparison.py": "d0a0295cbf034223b5de5ac47b7369f93115400fb2c15e0d81ca13b67919c01e", "competition/shell_interface_scoring_trial.py": "5df9920127442b922887178150fb6d682e6319208c38b28e91f7b48c774a30ee", "competition/development_comparison_checked.py": "d0f29c6db402553f0d873d42881e96d1c8992a780944d541bf071185d1d17830", "scripts/build_tool_path_comparison_notebooks.py": "6df902c98f4a9238cbfec83a8a6d94843b82abe2d0865b6c681375be0ba14115", "competition/declared_comparison_environment.py": "0e3b118f3ff82a8243e28729a20df0019027c1def6b039a2df2e43b8ab9d9eb9", "competition/offline_test_dependencies.py": "ef1f54a598380b8d233665893390b4e9f42fcc1cd02da5feec0ba9d317859327", "competition/supplemental_snapshot_dependencies.py": "101b94234a247535d904f6976860a1d7e42d9bd593b75eba982f552feb576fb1", "competition/fixture_integrity.py": "9a59465a39b3ad949b05896d64d37f73c5c66d25e60ab9f9b3ab746b99ffdfa6", "competition/frozen_agent_unpack.py": "d90d71ba52a118cd374a02d15149f01bd7a49dd902ab85a905152b8b2a342062", "competition/development_comparison_outcome_aware.py": "d2d8fc6615101de2d30e8613f3274bf21ce6990d8980541c1f967446f704052d", "scripts/prepare_broader_scoring_replay.py": "c5767faad2dfd24d1f1e81998ede9335209698584d158f162d6190b5415e5a40", "scripts/prepare_budget_comparison.py": "c7df3329b214b93a238af766d0e213c57e43e299d60808bcdf210247a15002ff", "competition/development_comparison_arm_limits.py": "2428be88a89fc58b3c56e00105f14653cfda80aa3c42bad1b4a84943b91a431f", "competition/gpu_budget_compiler_preflight.py": "3e42d634b346a29b725df86be104a54591aa14d8e05b2c09c6d31e77b9f20d7f", "scripts/build_runtime_calibration_candidate.py": "8b7e8dfd3a90fc139cfd085e1b23ad8be4c41857bb0a42d4be4278e3038e7277", "scripts/build_five_minute_candidate.py": "7f779c17c78aa6e4c0bc901546639164554b4221ff1a8bc5d093d53ad393443c", "competition/source_validation_extension.py": "11fc4a1b34af15752bb69e798f04e4d76f0f8fa30da4109ea9465f3364b79995", "scripts/build_source_validation_candidate.py": "33e97c0620d5336eaa3500bf07e262035ecbce1fe459407226c794d85605f1d9", "scripts/build_focused_repair_candidate.py": "ef8b336807a85a0df3e990a3a5f591ee6c1d7b3cd4fec5f41378832e7e8b5d88", "competition/focused_repair_prompt.md": "42373511dd04c010351edf8abddc9b3dafb535bc412529fbba60ddad100e9743", "scripts/prepare_focused_repair_comparison.py": "422f7a155239b2f7fda933aa7af12c0cc39fa10c946aa8a9f2ace2c7adaaa07f", "competition/source_interface_scoring_trial.py": "deb0ec629fe3b6fa879931df77a68833a177ba377baebd09b492022ecac4e008", "tests/test_source_interface_package_boundary.py": "287b16f9669c132e8792c7ac5af81fa8bf0fd1f9f5f86ad47f1261838ce31729", "scripts/build_structured_repair_revision.py": "f123a0e7a073170ab2ee41d08f82eb3626216fe2e8abdec6cb484ad0c6078e8d", "competition/repair_workspace_skill_v19.py": "d4c232d1301d8030c539702f5be2d1db83f5b0397a0cddd3e8d60dfaa6acd3d7", "competition/structured_repair_prompt.md": "c154754ab44b6b99dc558eded7a9e62a8863c0c4f2c8ffcf8ecf0179cabcc6e4", "competition/structured_repair_SKILL_v19.md": "f4c517b2335238d1f63d3982def2062ab3031440eb8bc5a3e2d715d854ac661c", "tests/test_repair_workspace_skill_v19.py": "259d073aeef1e06fd201de990e545a54ad6247a1e53dd8e9b848dd9e536c1e29", "reports/competition_focus/repair_workspace_skill_v19_checks_20261007.xml": "a3376973ac791c08913fc8988adedc82143fd0885ad0977b796beda8ccdc94f5", "reports/competition_focus/overnight_terminal_audit_20261007.json": "ace5662d7a571dc2b54b89265a5552da56412e4026c6892eb1934ee03f7b0a06", "reports/kaggle_runs/20261007_structured_repair_native_v1_terminal_1130/reconciliation.json": "6c9e19bf00a08f81bb6ccdd851f33df10d3ca6e94fbb939987b5d4c0e8a1786c", "reports/competition_focus/adk_skill_executor_source_20261007/google/adk/tools/skill_toolset.py": "fa3267855239bf6951ddbd40671b5385c6f60962ee4fbd5b9ac04b51c53a9fc6", "scripts/prepare_structured_repair_comparison.py": "b6ec033d1ef052b38e65f67127df7c75cfed34866f1b7f0a096f1f4e29c311c1", "competition/structured_interface_scoring_trial.py": "f22a2e35f9e4371210627c6dbebab13000fb9eb3f09c79a226a71663083faf08", "tests/test_structured_interface_package_boundary.py": "1c91642bb4c51825cfdb4b94ed26a308687e56cb9b76e75e08a31e2d6def81b6", "reports/competition_focus/structured_interface_package_checks_20261007.xml": "47b8428c30b79df324fe6fa7904950205fdaf86f14c6edffc20d412fa383da92", "scripts/build_simple_repair_candidate.py": "9c649579a127cc8d9f3b99891e2fd5ad122415e7d1eb2e28db88f743f8e90ef0", "competition/simple_repair_workspace_skill_v20.py": "9c3125cd9c3bfaf45896cc56dd0845cfd90013f1fc22f504b777e2c3b0bcccf8", "competition/simple_repair_prompt_v20.md": "a04e29114e1018d179c92a6186c66e02e2468c8cc614ddaaab5ec897d522b613", "competition/simple_repair_SKILL_v20.md": "3a765e1579f695a90e53deb0862093e84b214505d994caa020aeb392f0cadc5b", "tests/test_repair_workspace_skill_v20.py": "27293b4fd36b30b995df1117ad92665f488515a605f716caf40a3397c26da4ab", "reports/competition_focus/repair_workspace_skill_v20_checks_20261007.xml": "ccb1a42b10878a29907374d839aa8de22e43a076265ac9016f7447e38ad46bd7", "reports/competition_focus/structured_repair_terminal_audit_20261007.json": "cc5a45e4d451ccd2311a29951924417552cf73587baa5f900560659bd6578e2b", "reports/kaggle_runs/20261007_structured_repair_native_v2_terminal/reconciliation.json": "e982ad1931516a99e7b9933dfd0d3d1c64838343b2a373f7cb88c7cf5a998411", "scripts/prepare_simple_repair_comparison.py": "350788ffcd850409cdceaed0fb416bccae0f1f1dca169e892410fe28f6590e9a", "tests/test_simple_interface_package_boundary.py": "cc2b571a097beb93068237dcee89c350e329e5ad312c3b22692964553e443bd4", "reports/competition_focus/simple_interface_package_checks_20261007.xml": "ce3d4c8766702e68e70b15eaf2fd721a36dc272c94919ee226690cf3b0bbd236", "scripts/build_hypothesis_pipeline_candidate.py": "c51f5f7e90adafcc89e8fc621dcc55a80303f6762fe0d099c5cde763b1a49a32", "competition/hypothesis_planner_v21.md": "85ea8411a708049eee9a3c9317c8045ede88244a4e29144fc4cfb572b1dad4aa", "reports/competition_focus/source_validation_candidate_preparation_20261006.json": "1364896b1c8bf946968a54b058e8cefd43650652a2dcd6b1db9251b543235358", "reports/competition_focus/simple_repair_protocol_diagnosis_20261007.json": "3b3d96cede6d5927cd980cced88fd8e09e2b61aefda0e811d64d209c4104b363", "reports/competition_focus/simple_repair_candidate_selection_20261007.json": "998408574928b8d6841055d2a23d683327b300d0a993faf3f303113628750528", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/schema.py": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/builders/workflows.py": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/builders/sub_agents.py": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/harness/agent_runner.py": "dff7d2505340633c3fe50ba7073f2609fb2b41becd69c356d9af25cc4c1f98dd", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/config.py": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/models/discovery.py": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca", "benchmarks/competition/development/20261007_simple_repair_native_v1/plan.json": "180c3f23caaf06aca1ff4d4082cedb1e45378e99526f99f7f2e57ba1d59d19ff", "scripts/prepare_additional_fixture_panel.py": "bcaaab6ec65f011e53abbe289276b1620809d0c4c59c6815bff6e29261e18afd", "competition/private_fixture_diagnostics.py": "e9d82e697a31f4c2b4c122e2787d9712f8f08d20fca3a5f4f963148291b6dc87", "competition/public_fixture_controls.py": "6a1fe062faf69188e584bcbd8ffa6147e64b6bb65bd2588f5c2fb46beedd4af8", "scripts/prepare_hypothesis_pipeline_comparison.py": "1e668affd8bdb8a7485f96dfa3da95f2cf2a32387b7f381ba459c50e7be7483e", "competition/hypothesis_pipeline_scoring_trial.py": "cfd7d04b88e351e98caf58f351fe295ab066d56bf79e5aba65f9967e1277aa7a", "reports/kaggle_runs/20261007_hypothesis_native_terminal_1643/reconciliation.json": "f49761b76130db75af4d5b2128f57aaff656b2a469686ea45f0fbd3ac69cdf33", "scripts/reconcile_hypothesis_pipeline_native_probe.py": "95a2a5abeca8325cb6ddacf221c825c8d077947143241f3ef6845d24e6ff8ada", "reports/kaggle_runs/20261007_additional_fixture_terminal_1617/terminal_reconciliation.json": "20726a8141bbd387b8467756aefadcd52c023617474e15133a1a6f15edba24d4", "reports/competition_focus/additional_fixture_panel_preparation_20261007.json": "8ae05bc2b2a0dfa7316908594f2c230443b56bd3854ffd718b66fcd0de2b5c25", "tests/test_hypothesis_pipeline_package_boundary.py": "913d7d7516e5b128565c974e89e080cae9686aa906768e383db9f8e3661c3be8", "reports/competition_focus/hypothesis_pipeline_package_checks_20261007.xml": "13618440dc2dd03d2601960a8949fc956c06cab55d97662cc178c6b482a22536", "reports/competition_focus/hypothesis_pipeline_candidate_preparation_20261007.json": "d0221d8ba0e4325548b057cea9efc595aa993fffed9ba76a86e99853ae81bf7a", "competition/search_scope_v22.md": "fe04583dd97352806322aa667c27bd522b565d8ee5b1de0b8ded58dbc21eede9", "reports/kaggle_runs/20261007_native_search_scope_terminal_1817/reconciliation.json": "1e8a6ce050e611b75e6be36772621bc3b1abd5bcbe21d564398fa7f35adf6962", "scripts/build_search_scope_candidate.py": "cf8d28d3e8e69d6bf7ef84c59f085becf4b6d5e1aa975816183c05781abeacfe", "competition/native_search_scope_panel.py": "90a142207798d2687e56d4850604b63050913a7653b26d2a8be8c4e4824b1371", "scripts/reconcile_native_search_scope_probe.py": "c14d92065f23575a0e415ab095cb426b8180c3631825566d0a87cfa7e1c15397", "reports/competition_focus/hypothesis_pipeline_candidate_selection_20261007.json": "8300e917a95cf758a79f9ac3ed99fbbb128c2ec2e0db587331db6e3603bcef9b", "scripts/prepare_search_scope_comparison.py": "74b4438f89038a0ae417f7dbc38b20c515287339a3a27f54e85d364e95265059", "reports/kaggle_runs/20261007_search_scope_candidate_native_terminal_1834/reconciliation.json": "d598fc05bd2108f56e7d0c550007d745d8c88e80ab66d6419daf623d0abc732f", "scripts/reconcile_search_scope_candidate_native.py": "430aada64728af65be0dbb66920cf7ef72bed416342756d7404d9e53adb67c22", "tests/test_search_scope_package_boundary.py": "3ce22d60c77ec829c8d435d139442e43f89763c5dfa1ceec97b6a2779110eff5", "reports/competition_focus/search_scope_package_checks_20261007.xml": "f4dd436aa2b9b3e9aba1dc9a675a3054e7668ef7dc6bc6e4f697f0e1f28ce586", "reports/kaggle_runs/20261007_hypothesis_pipeline_terminal_1755/hypothesis_pipeline_terminal_audit_v2.json": "b368063fbe9d80a45f0e257cb9313879d4e29e5b459d798a6ce33262375a9ce4", "scripts/build_reasoning_budget_candidate.py": "2a3c0abdaaa20cdac726051ab99aa0d54c271da224b805f4b4957814cbc418a8", "reports/competition_focus/search_scope_terminal_timing_diagnosis_20261008.json": "5760d067cf2f248d41fc6dbb40ff3882579bbb366886b083fbdef031f49f9625", "reports/competition_focus/search_scope_candidate_selection_20261008.json": "6847af2cb332eb5cbac78105886d49453d4bdc44aa0f225db7f6745ae81e1fa0", "reports/competition_focus/vllm0191_budget_primary_20261008/source_receipt.json": "3aafefdc480e4cb02541009b50aa2188049b81a50d3c6744e36b05d5581d537f", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/server.py": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/resolvers/generation.py": "ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/config/reasoning.py": "8bc102f01f330fb2509b4fca6b5dc102eb86ccf791362e847045d76778565cb3", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/reasoning/gemma4_reasoning_parser.py": "64a1d3f05e22cee893ebc1f2507a150eaadf4d06e3efac5035c6ab94480a739e", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/entrypoints/openai/chat_completion/protocol.py": "1a2492bdb99e944ca177853496408a2a00d578bb2fa51c6e49241567d6601a82", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/v1/sample/logits_processor/builtin.py": "769038f9f91a613f2157556e74413466218e1ca9a5e9e166487a3660809d6e28", "scripts/prepare_reasoning_budget_comparison.py": "a4106a3a40a912a07471ce23ce3676a63c30cc707ac18a235a7bf7ee74bd1032", "reports/competition_focus/reasoning_budget_candidate_preparation_20261008.json": "4d8d044d9fcc8fc225658ddbfede021d3846ab8d3f806a65684e1939540a8ece", "reports/kaggle_runs/20261008_reasoning_budget_native_terminal_0524/reconciliation.json": "170e4ad0e66909666f0bde6fc774199019e99af28bde16e24c996b166b4cca76", "reports/competition_focus/reasoning_budget_native_preparation_20261008.json": "858542fa651d0996d7f18e2003d9a5c9807944749fd79d04eca3c78e641d1a7a", "scripts/audit_reasoning_budget_comparison_terminal.py": "88e7c3cd3679976090c2975d4eb852fd97465ae54d8584786bd5747bc00dfa69", "scripts/reconcile_reasoning_budget_native.py": "50b82e2eadd400dfb5f4acf585af1356d251c52be44d766dc9a142c600661208", "competition/native_generation_wire_panel.py": "657aedafca3ce6a836a4201aff86d07dba68ce3456c09273eacd5c68235dd841", "reports/competition_focus/terminal_reasoning_interval_review_20261008.json": "1a38605b545e46b18b7aa4cae0f817e0ad088debf9526134b130e7599ca6b943", "reports/competition_focus/reasoning_budget_wire_boundary_checks_20261008.xml": "4a29c562a8a78f3fda913da0771ccd07f10b09ab69bf78cec88941b7d41febd8", "tests/test_reasoning_budget_wire_boundary.py": "070db2e2903d8030387bc59e74b424b2d0cb4100b4a60eea84c5f6613a012f4f", "reports/competition_focus/reasoning_budget_terminal_boundary_checks_20261008.xml": "e2737f783455d32142576a0924e4f54cdf202e5f2b4c7cbd9ace8e8d7c1742f6", "tests/test_reasoning_budget_terminal_boundary.py": "da724fa3aee9d4ebe95bd9156beb5de84b34153c6b12747ae4cb0543e29d82d1", "reports/competition_focus/reasoning_budget_GPU_planning_quota_20261008.json": "8f0642a5123151aeef43f75ec91cc532c4728e11b9441b5373c2b2f51f38ecbd"}, "selected_case_ids": ["fastapi_14962", "fastapi_14099", "fastapi_14978", "rich_3043", "rich_3518", "rich_4076"], "baseline_preparation_sha256": "d0221d8ba0e4325548b057cea9efc595aa993fffed9ba76a86e99853ae81bf7a", "candidate_preparation_sha256": "4d8d044d9fcc8fc225658ddbfede021d3846ab8d3f806a65684e1939540a8ece", "compiler_parent_notebook_sha256": "3ccb40baa60e188e7a0b34645e17d87366ce549db0eeeccd77073bc94af64029", "native_candidate_reconciliation_sha256": "170e4ad0e66909666f0bde6fc774199019e99af28bde16e24c996b166b4cca76", "CPU63_reconciliation_sha256": "20726a8141bbd387b8467756aefadcd52c023617474e15133a1a6f15edba24d4"}')
AGENT_INPUT_PACKET_BASE64 = 'eyJiYXNlX2NvbW1pdCI6ICJjZmYyMjM2ZGFjYjdhZWRjYWYyZDJkZTA4NzNjZmJjOWJiMmI4OTdhIiwgImhpbnRzX3RleHQiOiAiIiwgImluc3RhbmNlX2lkIjogImZhc3RhcGlfMTQ0MzAiLCAicHJvYmxlbV9zdGF0ZW1lbnQiOiAi8J+QmyBGaXggb3B0aW9uYWwgc2VxdWVuY2UgaGFuZGxpbmcgd2l0aCBuZXcgdW5pb24gc3ludGF4IGZyb20gUHl0aG9uIDMuMTBcblxuaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9wdWxsLzE0Mjk3IGlzIG9ubHkgdGFraW5nIG9uZSB1bmlvbiB0eXBlIGludG8gYWNjb3VudC4gSSBjYW4gb25seSByZWNvbW1lbmQgdXNpbmcgaHR0cHM6Ly9naXRodWIuY29tL3B5ZGFudGljL3R5cGluZy1pbnNwZWN0aW9uIGZvciB0aGVzZSBraW5kIG9mIGNoZWNrcyAoc2VlIGh0dHBzOi8vdHlwaW5nLWluc3BlY3Rpb24ucHlkYW50aWMuZGV2L2xhdGVzdC91c2FnZS8gZm9yIG1vdGl2YXRpb24pLiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiZDJjMTdiNjAzZDJmOTg2ZDVmMzgwMGQzYmYyNGFjYmE4ZTFmNDg1YiIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0OTUzIiwgInByb2JsZW1fc3RhdGVtZW50IjogIuKZu++4jyBGaXggSlNPTiBTY2hlbWEgZm9yIGJ5dGVzLCB1c2UgYFwiY29udGVudE1lZGlhVHlwZVwiOiBcImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbVwiYCBpbnN0ZWFkIG9mIGBcImZvcm1hdFwiOiBcImJpbmFyeVwiYFxuXG7imbvvuI8gRml4IEpTT04gU2NoZW1hIGZvciBieXRlcywgdXNlIGBcImNvbnRlbnRNZWRpYVR5cGVcIjogXCJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW1cImAgaW5zdGVhZCBvZiBgXCJmb3JtYXRcIjogXCJiaW5hcnlcImBcclxuXHJcbiMjIEJhY2tncm91bmRcclxuXHJcbmBmb3JtYXQ6IGJpbmFyeWAgd2FzIGRlZmluZWQgaW4gT3BlbkFQSSAzLjAueCwgaW4gT3BlbkFQSSAzLjEueCB0aGUgc2NoZW1hIHdhcyBhbGlnbmVkIHdpdGggdGhlIGxhdGVzdCBKU09OIFNjaGVtYSwgcmVjb21tZW5kaW5nIGluc3RlYWQgYGNvbnRlbnRNZWRpYVR5cGU6IGFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbWAuXHJcblxyXG5JIHN1c3BlY3QgdGhlIEpTT04gU2NoZW1hIGZvciBgYnl0ZXNgIHVzaW5nIGBcImZvcm1hdFwiOiBcImJpbmFyeVwiYCBjb21lcyBmcm9tIG15IGZpcnN0IGltcGxlbWVudGF0aW9uIGluIFB5ZGFudGljIDEueC5cclxuXHJcbkl0IHdhcyBkZWZpbmVkIGFuZCBzdWdnZXN0ZWQgaW4gT3BlbkFQSSAzLjAueCAobm90IGluIEpTT04gU2NoZW1hKTogaHR0cHM6Ly9zcGVjLm9wZW5hcGlzLm9yZy9vYXMvdjMuMC4zLmh0bWwjY29uc2lkZXJhdGlvbnMtZm9yLWZpbGUtdXBsb2Fkc1xyXG5cclxuT3BlbkFQSSAzLjEueCBhbGlnbmVkIHN1cHBvcnQgd2l0aCBKU09OIFNjaGVtYSBkcmFmdCAwNywgc28gaXQgd2FzIHN1Z2dlc3RlZCB0byB1cGF0ZSBmaWxlIHVwbG9hZHMgdG8gdXNlIHRoZSByZWd1bGFyIEpTT04gU2NoZW1hIGZvcm1hdDogYFwiY29udGVudE1lZGlhVHlwZVwiOiBcImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbVwiYDogaHR0cHM6Ly9sZWFybi5vcGVuYXBpcy5vcmcvdXBncmFkaW5nL3YzLjAtdG8tdjMuMVxyXG5cclxuVGhpcyBpcyBkZWZpbmVkIGluIEpTT04gU2NoZW1hIDA3OiBodHRwczovL2pzb24tc2NoZW1hLm9yZy9kcmFmdC0wNy9qc29uLXNjaGVtYS12YWxpZGF0aW9uI3JmYy5zZWN0aW9uLjguNFxyXG5cclxuIyMjIEpTT04gU2NoZW1hIDIwMjAtMTIgTm90ZVxyXG5cclxuTm93IE9wZW5BUEkgMy4yIGlzIGFsaWduZWQgd2l0aCBKU09OIFNjaGVtYSAyMDIwLTEyLCB3aGljaCBpcyB3aGF0IFB5ZGFudGljIHYyIGltcGxlbWVudHMgKGV4Y2VwdCBmb3IgdGhpcywgSSdtIGltcGxlbWVudGluZyBpdCB0aGVyZSB0b28pLlxyXG5cclxuSXQncyB0aGUgc2FtZSBhcyBpbiBKU09OIFNjaGVtYSBkcmFmdCAwNywgc28gdGhpcyBzdGlsbCBhcHBsaWVzOiBodHRwczovL2pzb24tc2NoZW1hLm9yZy9kcmFmdC8yMDIwLTEyL2RyYWZ0LWJodXR0b24tanNvbi1zY2hlbWEtdmFsaWRhdGlvbi0wMSNuYW1lLWNvbnRlbnRtZWRpYXR5cGVcclxuXHJcbiMjIyBVc2FnZSBpbiBKU09OXHJcblxyXG5KU09OIGFzIGEgZm9ybWF0IGFjdHVhbGx5IGRvZXNuJ3Qgc3VwcG9ydCBieXRlcywgZXZlcnl0aGluZyBoYXMgdG8gYmUgaW4gVVRGLTggc3RyaW5ncy4gVHJhbnNwb3J0aW5nIGJ5dGVzIGluIEpTT04gd291bGQgcmVxdWlyZSBlbmNvZGluZyBieXRlcyBpbiBhIHN0cmluZywgZS5nLiB3aXRoIGJhc2U2NC5cclxuXHJcbkJ1dCBhcyBKU09OIFNjaGVtYSBpcyBub3Qgb25seSBkZWZpbmVkIHRvIGRlY2xhcmUgSlNPTiBwYXlsb2FkcyBidXQgYWxzbyBwYXlsb2FkcyB0aGF0IGNvdWxkIGhhdmUgYSBjb21wYXJhYmxlIHN0cnVjdHVyZSBhbmQgZGVmaW5lZCB3aXRoIEpTT04gU2NoZW1hLCBpdCdzIHN0aWxsIHRoZXJlIGluIHRoZSBzcGVjLiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiNzFhMTdiNTkzMmUzOGJjZjc5OGZjZDA1YzViYjM4NGU2ZDRkYjViZiIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0NDgyIiwgInByb2JsZW1fc3RhdGVtZW50IjogIvCfkJsgRml4IGhhbmRsaW5nIGFyYml0cmFyeSB0eXBlcyB3aGVuIHVzaW5nIGBhcmJpdHJhcnlfdHlwZXNfYWxsb3dlZD1UcnVlYFxuXG7wn5CbIEZpeCBoYW5kbGluZyBhcmJpdHJhcnkgdHlwZXMgd2hlbiB1c2luZyBgYXJiaXRyYXJ5X3R5cGVzX2FsbG93ZWQ9VHJ1ZWBcclxuXHJcblJlbGF0ZWQgdG8gaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9kaXNjdXNzaW9ucy8xNDE4NCwgaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9pc3N1ZXMvMTQ0ODNcclxuXHJcbiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiNTI2NWM0ZjVjYjRkOGZiNGZiNTAxNWJjY2YxYTZkZThmM2ZjNjA0NCIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0MzYxIiwgInByb2JsZW1fc3RhdGVtZW50IjogIvCfkJsgQXZvaWQgYWNjZXNzaW5nIG5vbi1leGlzdGluZyBcIiRyZWZcIiBrZXkgZm9yIFB5ZGFudGljIHYyIGNvbXBhdCByZW1hcHBpbmdcblxuY2YgRGlzY3Vzc2lvbiAjMTQyNjUiLCAicmVwbyI6ICJmYXN0YXBpL2Zhc3RhcGkifQp7ImJhc2VfY29tbWl0IjogIjRkNmQ2MzFhM2QyZGVkZGY4NDA1NTIyZDRiOGM5NzZhNmQzNTcyNmMiLCAiaGludHNfdGV4dCI6ICIiLCAiaW5zdGFuY2VfaWQiOiAicmljaF8zODgyIiwgInByb2JsZW1fc3RhdGVtZW50IjogIkZpeCByYXcgbWFya3VwIHByaW50ZWQgb24gcHJvbXB0IGVycm9ycyBzb21ldGltZXNcblxuIyMgVHlwZSBvZiBjaGFuZ2VzXHJcblxyXG4tIFt4XSBCdWcgZml4XHJcbi0gWyBdIE5ldyBmZWF0dXJlXHJcbi0gWyBdIERvY3VtZW50YXRpb24gLyBkb2NzdHJpbmdzXHJcbi0gWyBdIFRlc3RzXHJcbi0gWyBdIE90aGVyXHJcblxyXG4jIyBDaGVja2xpc3RcclxuXHJcbi0gW3hdIEkndmUgcnVuIHRoZSBsYXRlc3QgW2JsYWNrXShodHRwczovL2dpdGh1Yi5jb20vcHNmL2JsYWNrKSB3aXRoIGRlZmF1bHQgYXJncyBvbiBuZXcgY29kZS5cclxuLSBbIF0gSSd2ZSB1cGRhdGVkIENIQU5HRUxPRy5tZCBhbmQgQ09OVFJJQlVUT1JTLm1kIHdoZXJlIGFwcHJvcHJpYXRlIChzZWUgbm90ZSBhYm91dCB0eXBvcyBhYm92ZSkuXHJcbi0gW3hdIEkndmUgYWRkZWQgdGVzdHMgZm9yIG5ldyBjb2RlLlxyXG4tIFt4XSBJIGFjY2VwdCB0aGF0IEB3aWxsbWNndWdhbiBtYXkgYmUgcGVkYW50aWMgaW4gdGhlIGNvZGUgcmV2aWV3LlxyXG5cclxuIyMgRGVzY3JpcHRpb25cclxuXHJcblByb21wdEJhc2Uub25fdmFsaWRhdGVfZXJyb3IoKSBzaG91bGQgZXhwbGljaXRseSBhc2sgZm9yIG1hcmt1cCBpbiBjYXNlIHRoZSBjb25zb2xlIHBhc3NlZCB0byBpdCBoYXMgbWFya3VwIGRpc2FibGVkIGdsb2JhbGx5LlxyXG5cclxuRml4ZXMgIzM4ODEuIiwgInJlcG8iOiAiVGV4dHVhbGl6ZS9yaWNoIn0KeyJiYXNlX2NvbW1pdCI6ICI0MTAxOTkxODk4ZWU3YTA5ZmUxNzA2ZGFjYTI0YWY1ZTFlMDU0ODYyIiwgImhpbnRzX3RleHQiOiAiIiwgImluc3RhbmNlX2lkIjogInJpY2hfMzUwNiIsICJwcm9ibGVtX3N0YXRlbWVudCI6ICJmaXggc3BsaXQgY2VsbHNcblxuRml4ZWQgYFNlZ21lbnQuX3NwbGl0X2NlbGxzYCBpc3N1ZSBkaXNjb3ZlcmVkIGluIGh0dHBzOi8vZ2l0aHViLmNvbS9UZXh0dWFsaXplL3RleHR1YWwvaXNzdWVzLzQ5OTYiLCAicmVwbyI6ICJUZXh0dWFsaXplL3JpY2gifQo='
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('writer_reasoning_budget70_' + MODE + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen_plan_bytes = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen_plan_bytes).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen_plan_bytes) != PLAN:
        raise ValueError('Frozen plan bytes differ from source binding')
    (RUN / 'plan.json').write_bytes(frozen_plan_bytes)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    packet = base64.b64decode(AGENT_INPUT_PACKET_BASE64, validate=True)
    if hashlib.sha256(packet).hexdigest() != PLAN['selected_problem_only_input_sha256']:
        raise ValueError('Problem-only input differs from the frozen plan')
    (RUN / 'agent_inputs.jsonl').write_bytes(packet)
    unpack_frozen_agents(RUN, PLAN, PACKAGES)
    export_run(RUN)
    print('Frozen output directory:', RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    if MODE == 'gpu_comparison':
        if not MODEL_PATH.is_dir():
            raise ValueError('Attach exact 31B QAT revision 2 model')
        for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
            'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
            'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
            'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
            os.environ[key] = value
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
    if hashlib.sha256((DATA_DIR / 'tasks.jsonl').read_bytes()).hexdigest() != PLAN['tasks_sha256']:
        raise ValueError('Official task revision differs from the frozen plan')
    if hashlib.sha256((DATA_DIR / 'HARNESS_README.md').read_bytes()).hexdigest() != PLAN['guide_sha256']:
        raise ValueError('Official guide revision differs from the frozen plan')
    from swegemma.models import load_tasks
    actual = {task.instance_id: task for task in load_tasks(DATA_DIR / 'tasks.jsonl')}
    for line in (RUN / 'agent_inputs.jsonl').read_text(encoding='utf-8').splitlines():
        problem = json.loads(line)
        if set(problem) != {'instance_id', 'repo', 'base_commit', 'problem_statement', 'hints_text'}:
            raise ValueError('Problem-only input includes unexpected fields')
        if any(getattr(actual[problem['instance_id']], name) != value for name, value in problem.items()):
            raise ValueError('Actual organizer problem-only input differs from frozen input')


In [ ]:
"""Install genuine test dependencies offline without replacing the verifier."""
import hashlib
import importlib
import importlib.metadata as metadata
import io
import json
from pathlib import Path
import platform
import subprocess
import sys
import zipfile


def verify_dependency_bundle(payload: bytes, manifest: dict) -> dict[str, bytes]:
    if hashlib.sha256(payload).hexdigest() != manifest['bundle_sha256']:
        raise ValueError('Dependency bundle hash mismatch')
    expected = {row['file']: row for row in manifest['files']}
    if len(expected) != len(manifest['files']):
        raise ValueError('Duplicate dependency manifest member')
    if any(Path(name).name != name or '/' in name or '\\' in name or not name.endswith('.whl')
           for name in expected):
        raise ValueError('Dependency wheels must be plain basenames')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dependency archive/manifest member mismatch')
        if sum(item.file_size for item in archive.infolist()) > 16 * 1024 * 1024:
            raise ValueError('Unexpected dependency archive size')
        result = {}
        for item in archive.infolist():
            if item.is_dir() or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Invalid dependency archive member')
            data = archive.read(item.filename)
            row = expected[item.filename]
            if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                raise ValueError('Dependency wheel identity mismatch: ' + item.filename)
            result[item.filename] = data
        return result


def install_offline_test_dependencies(payload: bytes, manifest: dict, run_root: Path, checkpoint_exporter) -> dict:
    if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
        raise ValueError('This dependency bundle targets the Kaggle Linux x86_64 runtime')
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise ValueError('This bundle contains wheels for Python 3.12 and 3.13 only')
    wheels = verify_dependency_bundle(payload, manifest)
    profiles = {
        'httpbin_current': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4'],
        'httpbin_legacy_redirect': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4',
                                   'Flask==2.2.5', 'Werkzeug==2.2.3'],
    }
    profile = manifest.get('profile', 'httpbin_current')
    requirements = profiles.get(profile)
    if requirements is None or manifest['requirements'] != requirements:
        raise ValueError('Unexpected test-environment requirements')
    stage = Path('/tmp') / (run_root.name + '_dependency_wheels')
    stage.mkdir(exist_ok=False)
    for name, data in wheels.items():
        (stage / name).write_bytes(data)
    packages = sorted({row['package'] for row in manifest['files']} |
                      {'swegemma', 'adk-submission', 'adk-eval-core', 'vllm', 'torch', 'pytest'})
    def versions():
        result = {}
        for name in packages:
            try:
                result[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                result[name] = None
        return result
    before = versions()
    report = {'scope':'supplemental_offline_test_environment_not_agent_or_private_scoring',
              'python_version':sys.version, 'platform':platform.platform(),
              'bundle_sha256':manifest['bundle_sha256'], 'requirements':requirements, 'profile':profile,
              'before':before, 'model_inference_executed':False,
              'organizer_verifier_replaced':False, 'test_assertions_changed':False}
    (run_root / 'dependency_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
    process = subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        '--no-index','--find-links',str(stage),*requirements],capture_output=True,text=True)
    (run_root / 'dependency_install_stdout.txt').write_text(process.stdout,encoding='utf-8')
    (run_root / 'dependency_install_stderr.txt').write_text(process.stderr,encoding='utf-8')
    importlib.invalidate_caches()
    after = versions()
    report.update(return_code=process.returncode,after=after,
        changed_packages={name:{'before':before[name],'after':after[name]}
                          for name in packages if before[name]!=after[name]},
        pytest_entry_points=[{'name':entry.name,'value':entry.value,'distribution':entry.dist.name}
                             for entry in metadata.entry_points(group='pytest11')])
    core_names = ('swegemma','adk-submission','adk-eval-core','vllm','torch','pytest')
    report['organizer_core_versions_unchanged'] = all(before[name]==after[name] for name in core_names)
    report['httpbin_plugin_registered'] = any(entry['value']=='pytest_httpbin.plugin'
                                            for entry in report['pytest_entry_points'])
    report['status'] = ('installed' if process.returncode==0 and report['organizer_core_versions_unchanged']
                        and report['httpbin_plugin_registered'] else 'installation_not_verified')
    (run_root/'dependency_install_report.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
    checkpoint_exporter(run_root)
    if report['status'] != 'installed':
        raise RuntimeError('Offline dependency installation not verified; preserve its real logs')
    print('OFFLINE_TEST_DEPENDENCIES ' + json.dumps(report),flush=True)
    return report

"""Install a frozen genuine snapshot plugin without changing the organizer core."""
import base64
import hashlib
import importlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys
import tempfile


def install_snapshot_dependencies(manifest, encoded_wheels, run_root, checkpoint_exporter):
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    if sys.platform != 'linux' or sys.version_info[:2] != (3, 12):
        raise ValueError('This separately frozen environment requires Linux CPython3.12')
    if {r['file'] for r in manifest['files']} != set(encoded_wheels):
        raise ValueError('Snapshot dependency membership differs')
    core = ('swegemma', 'adk-submission', 'adk-eval-core', 'vllm', 'torch', 'pytest', 'rich')
    names = sorted(set(core) | {r['package'] for r in manifest['files']})
    def versions():
        result = {}
        for name in names:
            try:
                result[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                result[name] = None
        return result
    before = versions()
    report = {'scope': 'supplemental_snapshot_dependencies_not_agent_or_private_scoring',
              'before': before, 'requirements': manifest['requirements'],
              'organizer_verifier_replaced': False, 'test_assertions_changed': False,
              'model_inference_executed': False}
    with tempfile.TemporaryDirectory(prefix='neural_swe_snapshot_wheels_') as temporary:
        folder = Path(temporary)
        paths = []
        for row in manifest['files']:
            name = row['file']
            if Path(name).name != name or '/' in name or '\\' in name or not name.endswith('.whl'):
                raise ValueError('Snapshot wheel must be a plain filename')
            data = base64.b64decode(encoded_wheels[name], validate=True)
            if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                raise ValueError('Snapshot wheel differs from its frozen PyPI identity')
            path = folder / name
            path.write_bytes(data)
            paths.append(str(path))
        process = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
            '--no-deps', '--force-reinstall', *paths], capture_output=True, text=True, timeout=180)
    (run_root / 'snapshot_dependency_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n', encoding='utf-8')
    (run_root / 'snapshot_install_stdout.txt').write_text(process.stdout or '', encoding='utf-8')
    (run_root / 'snapshot_install_stderr.txt').write_text(process.stderr or '', encoding='utf-8')
    importlib.invalidate_caches()
    after = versions()
    unsatisfied = []
    for row in manifest['files']:
        if after[row['package']] != row['version']:
            unsatisfied.append(row['package'] + ' version differs')
        for text in row['requires_dist']:
            requirement = Requirement(text)
            if requirement.marker is not None and not requirement.marker.evaluate({'extra': ''}):
                continue
            try:
                installed = metadata.version(canonicalize_name(requirement.name))
            except metadata.PackageNotFoundError:
                installed = None
            if installed is None or installed not in requirement.specifier:
                unsatisfied.append(text)
    unchanged = all(before[name] == after[name] for name in core)
    report.update(return_code=process.returncode, after=after,
        changed_packages={name: {'before': before[name], 'after': after[name]}
                          for name in names if before[name] != after[name]},
        organizer_core_versions_unchanged=unchanged, unsatisfied_requirements=unsatisfied,
        status='installed' if process.returncode == 0 and unchanged and not unsatisfied else 'installation_not_verified')
    (run_root / 'snapshot_dependency_install_report.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    checkpoint_exporter(run_root)
    if report['status'] != 'installed':
        raise RuntimeError('Snapshot dependency installation is not verified; preserve receipts')
    return report


In [ ]:
DEPENDENCY_MANIFEST = json.loads('{"scope": "supplemental_test_dependencies_not_agent_submission", "created_utc": "2026-10-03T19:45:18.893670+00:00", "profile": "httpbin_legacy_redirect", "requirements": ["pytest-httpbin==2.1.0", "httpbin==0.10.4", "Flask==2.2.5", "Werkzeug==2.2.3"], "target": "Linux x86_64 CPython 3.12/3.13", "bundle_sha256": "fe32aa50e4fb0f1c4886e2c320382744d6022d07296035e1b72259f7698769f2", "files": [{"file": "blinker-1.9.0-py3-none-any.whl", "package": "blinker", "version": "1.9.0", "bytes": 8458, "sha256": "ba0efaa9080b619ff2f3459d1d500c57bddea4a6b424b60a91141db6fd2f08bc", "source_url": "https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/blinker/1.9.0/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "brotlicffi", "version": "1.2.0.2", "bytes": 1536851, "sha256": "22916101de0e7ff535f2edf54b52a85591853b8ae9a98737643defdd3c063a3a", "source_url": "https://files.pythonhosted.org/packages/06/78/076419ed6c2c6aa3eaac6fd6b076502b4be89d50625fcdc513cd4aeca718/brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/brotlicffi/1.2.0.2/json", "requires_python": ">=3.8", "requires_dist": ["cffi>=1.0.0; python_version < \\"3.13\\"", "cffi>=1.17.0; python_version >= \\"3.13\\""]}, {"file": "cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221822, "sha256": "c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf", "source_url": "https://files.pythonhosted.org/packages/b1/db/dceb9dd5b231e1da801793f8acc9f3c52a7e1afe40bb1aae37e02b0faad5/cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221824, "sha256": "a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2", "source_url": "https://files.pythonhosted.org/packages/95/95/86342356ff5953b3fb06f7ef7c5bee212d45e770abc7218d451b9148313c/cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "click-8.5.0-py3-none-any.whl", "package": "click", "version": "8.5.0", "bytes": 125251, "sha256": "255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360", "source_url": "https://files.pythonhosted.org/packages/58/50/6c0d534c5f134586a8e1ba4e330569e32f057e33372ae556463212fb4cd3/click-8.5.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/click/8.5.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "decorator-5.3.1-py3-none-any.whl", "package": "decorator", "version": "5.3.1", "bytes": 10365, "sha256": "f47fe6fdbd2edd623ecfe36875d37aba411624e2670dd395dddae1358689bb3c", "source_url": "https://files.pythonhosted.org/packages/05/7f/798705f5296a58ca505d600456748d1be48078eac8a7050d8a98bc9edb89/decorator-5.3.1-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/decorator/5.3.1/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "Flask-2.2.5-py3-none-any.whl", "package": "Flask", "version": "2.2.5", "bytes": 101817, "sha256": "58107ed83443e86067e41eff4631b058178191a355886f8e479e347fa1285fdf", "source_url": "https://files.pythonhosted.org/packages/9f/1a/8b6d48162861009d1e017a9740431c78d860809773b66cac220a11aa3310/Flask-2.2.5-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Flask/2.2.5/json", "requires_python": ">=3.7", "requires_dist": ["Werkzeug (>=2.2.2)", "Jinja2 (>=3.0)", "itsdangerous (>=2.0)", "click (>=8.0)", "importlib-metadata (>=3.6.0) ; python_version < \\"3.10\\"", "asgiref (>=3.2) ; extra == \'async\'", "python-dotenv ; extra == \'dotenv\'"]}, {"file": "greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 621404, "sha256": "975736b002ed080d124cf81a79cb7e05cb26d6b3f5c7a7b651c0fcce70353aa1", "source_url": "https://files.pythonhosted.org/packages/66/6a/1594f3869c57c149abdb380492529e04d4c0229b5e4d79572c5bd0aaa673/greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 623779, "sha256": "f98e8215e172f567ce80eeaed9107fb4d32b6c44f26983d9b8334658136a205a", "source_url": "https://files.pythonhosted.org/packages/85/cb/ab0c123c514ed4e94c0dc9ee2e86362633e6b998cfc05de7fc9ac2eb9690/greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "httpbin-0.10.4-py2.py3-none-any.whl", "package": "httpbin", "version": "0.10.4", "bytes": 100765, "sha256": "71862093f371c3a39cfcdcd2f073aaf610b6433f9d4c0130248e676453fca65d", "source_url": "https://files.pythonhosted.org/packages/e5/39/e9fc7334c69f82c307ae861e13704437d31b71514d273061e08db5830b92/httpbin-0.10.4-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/httpbin/0.10.4/json", "requires_python": ">=3.8", "requires_dist": ["brotlicffi", "decorator", "flask>=2.2.4", "greenlet<3.0; python_version < \\"3.12\\"", "greenlet>=3.0.0a1; python_version >= \\"3.12.0rc0\\"", "importlib-metadata; python_version < \\"3.8\\"", "six", "werkzeug>=2.2.2", "pytest; extra == \\"test\\"", "tox; extra == \\"test\\"", "flasgger; extra == \\"mainapp\\"", "gunicorn; extra == \\"mainapp\\"", "gevent; extra == \\"mainapp\\""]}, {"file": "itsdangerous-2.2.0-py3-none-any.whl", "package": "itsdangerous", "version": "2.2.0", "bytes": 16234, "sha256": "c6242fc49e35958c8b15141343aa660db5fc54d4f13a1db01a3f5891b98700ef", "source_url": "https://files.pythonhosted.org/packages/04/96/92447566d16df59b2a776c0fb82dbc4d9e07cd95062562af01e408583fc4/itsdangerous-2.2.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/itsdangerous/2.2.0/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "jinja2-3.1.6-py3-none-any.whl", "package": "Jinja2", "version": "3.1.6", "bytes": 134899, "sha256": "85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67", "source_url": "https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Jinja2/3.1.6/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe>=2.0", "Babel>=2.7 ; extra == \\"i18n\\""]}, {"file": "markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 22985, "sha256": "8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581", "source_url": "https://files.pythonhosted.org/packages/4f/a7/aeedb5140afa41fc74c225e9184ab96723a6e873b6ee1c9fede7283456d8/markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 23017, "sha256": "434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6", "source_url": "https://files.pythonhosted.org/packages/63/e0/cec6865dfe88cb48fedd4b20aed6af5158e41092adcbf3e028bcc6ec2108/markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "pycparser-3.0-py3-none-any.whl", "package": "pycparser", "version": "3.0", "bytes": 48172, "sha256": "b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992", "source_url": "https://files.pythonhosted.org/packages/0c/c3/44f3fbbfa403ea2a7c779186dc20772604442dde72947e7d01069cbe98e3/pycparser-3.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pycparser/3.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "pytest_httpbin-2.1.0-py3-none-any.whl", "package": "pytest-httpbin", "version": "2.1.0", "bytes": 8309, "sha256": "b3bf7346cc2ad231447189cd85b458e341056684a7a69d69533dd29692209cdd", "source_url": "https://files.pythonhosted.org/packages/0d/97/149289c7123f75e98edc94872545c2f597f349ee037b1e0a47aef0362fd4/pytest_httpbin-2.1.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pytest-httpbin/2.1.0/json", "requires_python": ">=3.8", "requires_dist": ["httpbin", "requests; extra == \\"test\\"", "pytest; extra == \\"test\\""]}, {"file": "six-1.17.0-py2.py3-none-any.whl", "package": "six", "version": "1.17.0", "bytes": 11050, "sha256": "4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274", "source_url": "https://files.pythonhosted.org/packages/b7/ce/149a00dd41f10bc29e5921b496af8b574d8413afcd5e30dfa0ed46c2cc5e/six-1.17.0-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/six/1.17.0/json", "requires_python": ">=2.7, !=3.0.*, !=3.1.*, !=3.2.*", "requires_dist": []}, {"file": "Werkzeug-2.2.3-py3-none-any.whl", "package": "Werkzeug", "version": "2.2.3", "bytes": 233551, "sha256": "56433961bc1f12533306c624f3be5e744389ac61d722175d543e1751285da612", "source_url": "https://files.pythonhosted.org/packages/f6/f8/9da63c1617ae2a1dec2fbf6412f3a0cfe9d4ce029eccbda6e1e4258ca45f/Werkzeug-2.2.3-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Werkzeug/2.2.3/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe (>=2.1.1)", "watchdog ; extra == \'watchdog\'"]}], "verifier_or_test_assertion_changes": false, "dependency_rationale": "Werkzeug 2.2.3 preserves ASCII Location headers via safe_conversion; Flask 2.2.5 supports that version. The Python 3.12-compatible httpbin plugin remains unchanged. This is a supplemental compatibility profile, not the historical requirements lock or proof of equivalence to private scoring.", "source_references": ["https://raw.githubusercontent.com/psf/requests/0b4d494192de489701d3a2e32acef8fb5d3f042e/requirements-dev.txt", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/wrappers/response.py", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/urls.py"]}')
DEPENDENCY_INPUT_PREFIX = 'httpbin_legacy_redirect'
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    # Attach the private NeuralSWE Offline Test Dependencies dataset. Package
    # binaries are input files, not notebook source (Kaggle limits source to 1 MB).
    matches = list(Path('/kaggle/input').rglob(DEPENDENCY_INPUT_PREFIX+'_manifest.json'))
    if len(matches) != 1:
        raise RuntimeError('Attach exactly one version of the supplemental dependency dataset')
    input_root = matches[0].parent
    if json.loads(matches[0].read_text()) != DEPENDENCY_MANIFEST:
        raise ValueError('Attached dependency manifest differs from the frozen source')
    archive_path = input_root/(DEPENDENCY_INPUT_PREFIX+'_wheels.zip')
    if archive_path.is_file():
        dependency_payload = archive_path.read_bytes()
    else:
        # Kaggle may expand uploaded ZIPs. Recreate the same deterministic
        # transport archive from byte-verified original wheels without alteration.
        stream = io.BytesIO()
        with zipfile.ZipFile(stream,'w',compression=zipfile.ZIP_STORED) as archive:
            for row in DEPENDENCY_MANIFEST['files']:
                locations = list(input_root.rglob(row['file']))
                locations = [p for p in locations if
                    DEPENDENCY_INPUT_PREFIX+'_wheels' in p.parts]
                if len(locations) != 1:
                    raise ValueError('Missing or ambiguous genuine dependency wheel: '+row['file'])
                data = locations[0].read_bytes()
                if len(data)!=row['bytes'] or hashlib.sha256(data).hexdigest()!=row['sha256']:
                    raise ValueError('Attached dependency wheel changed')
                item=zipfile.ZipInfo(row['file'],date_time=(1980,1,1,0,0,0))
                item.create_system=0  # Match the recorded Windows transport metadata.
                item.external_attr=0o100644 << 16
                archive.writestr(item,data)
        dependency_payload = stream.getvalue()
    dependency_report = install_offline_test_dependencies(
        dependency_payload,DEPENDENCY_MANIFEST,RUN,export_run)
else:
    print('Local definitions only; no test-environment installation performed')

SNAPSHOT_MANIFEST = json.loads('{"scope": "genuine_supplemental_test_dependencies_not_agent_or_private_scoring", "files": [{"file": "asttokens-3.0.1-py3-none-any.whl", "package": "asttokens", "version": "3.0.1", "bytes": 27047, "sha256": "15a3ebc0f43c2d0a50eeafea25e19046c68398e487b9f1f5b517f7c0f40f976a", "source_url": "https://files.pythonhosted.org/packages/d2/39/e7eaf1799466a4aef85b6a4fe7bd175ad2b1c6345066aa33f1f58d4b18d0/asttokens-3.0.1-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/asttokens/3.0.1/json", "requires_dist": ["astroid<5,>=2; extra == \\"astroid\\"", "astroid<5,>=2; extra == \\"test\\"", "pytest<9.0; extra == \\"test\\"", "pytest-cov; extra == \\"test\\"", "pytest-xdist; extra == \\"test\\""]}, {"file": "executing-2.2.1-py2.py3-none-any.whl", "package": "executing", "version": "2.2.1", "bytes": 28317, "sha256": "760643d3452b4d777d295bb167ccc74c64a81df23fb5e08eff250c425a4b2017", "source_url": "https://files.pythonhosted.org/packages/c1/ea/53f2148663b321f21b5a606bd5f191517cf40b7072c0497d3c92c4a13b1e/executing-2.2.1-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/executing/2.2.1/json", "requires_dist": ["asttokens>=2.1.0; extra == \\"tests\\"", "ipython; extra == \\"tests\\"", "pytest; extra == \\"tests\\"", "coverage; extra == \\"tests\\"", "coverage-enable-subprocess; extra == \\"tests\\"", "littleutils; extra == \\"tests\\"", "rich; python_version >= \\"3.11\\" and extra == \\"tests\\""]}, {"file": "inline_snapshot-0.36.0-py3-none-any.whl", "package": "inline-snapshot", "version": "0.36.0", "bytes": 98033, "sha256": "1079cf302098fedb418c7ac2be9cadd14f2e71461143bcaf605be1214530bef4", "source_url": "https://files.pythonhosted.org/packages/39/de/f625b9c923c4264ee8d94e416c0d5fd59194f7b6cd8fc0d6ec2f51dfc506/inline_snapshot-0.36.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/inline-snapshot/0.36.0/json", "requires_dist": ["asttokens>=2.0.5", "executing>=2.2.0", "pytest>=8.3.4", "rich>=13.7.1", "tomli>=2.0.0; python_version < \'3.11\'", "typing-extensions", "black>=23.3.0; extra == \'black\'", "dirty-equals>=0.9.0; extra == \'dirty-equals\'"]}], "requirements": ["inline-snapshot==0.36.0", "asttokens==3.0.1", "executing==2.2.1"], "assertions_changed": false}')
SNAPSHOT_WHEELS = {'asttokens-3.0.1-py3-none-any.whl': '...', 'executing-2.2.1-py2.py3-none-any.whl': '...', 'inline_snapshot-0.36.0-py3-none-any.whl': '...'}
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    install_snapshot_dependencies(SNAPSHOT_MANIFEST, SNAPSHOT_WHEELS, RUN, export_run)


In [ ]:
"""A small, locked development comparison using the actual organizer evaluator.

Development results cannot be reported as held-out or leaderboard results. Every
assigned arm/task pair remains in the denominator, including interrupted runs.
"""
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import tempfile
import time


def assignment_key(row: dict) -> tuple:
    """Explicit trial IDs allow repeated assignments without merging outcomes."""
    values = (row['arm'], row['task_id'])
    for value in (*values, *([row['trial_id']] if 'trial_id' in row else [])):
        if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', value):
            raise ValueError('Assignment identifiers must be safe nonempty path components')
    return (*values, row.get('trial_id'))


def atomic_json(path: Path, value: dict) -> None:
    """A crash must leave the previous complete receipt or the next complete one."""
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n',
                                         dir=path.parent, prefix='.' + path.name + '.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(json.dumps(value, indent=2) + '\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def summarize(plan: dict, observations: list[dict]) -> dict:
    if not plan['schedule']:
        raise ValueError('A locked comparison must assign at least one trial')
    for arm in plan['arms']:
        assignment_key({'arm': arm, 'task_id': 'configuration'})
    expected = {assignment_key(row) for row in plan['schedule']}
    if len(expected) != len(plan['schedule']):
        raise ValueError('Duplicate planned assignment; repeats require distinct trial_id values')
    if any(pair[0] not in plan['arms'] for pair in expected):
        raise ValueError('Scheduled arm is not declared')
    seen, counts = set(), {arm: 0 for arm in plan['arms']}
    completed = 0
    for row in observations:
        key = assignment_key(row)
        if key not in expected or key in seen:
            raise ValueError('Unexpected or duplicate observed arm/task pair')
        seen.add(key)
        result = row.get('result', {})
        completed += int(result.get('status') == 'completed')
        passed = (result.get('status') == 'completed'
                  and result.get('official_resolved') is True
                  and result.get('test_exit_code') == 0
                  and result.get('patch_bytes', 0) > 0)
        counts[row['arm']] += int(passed)
    return {
        'scope': 'public_development_comparison_not_held_out_or_leaderboard',
        'assigned_pairs': len(expected), 'observed_pairs': len(seen),
        'completed_pairs': completed, 'incomplete_observed_pairs': len(seen) - completed,
        'unattempted_pairs': len(expected - seen),
        'arms': {arm: {'verified_resolved': counts[arm],
                      'assigned_count': sum(pair[0] == arm for pair in expected)}
                 for arm in plan['arms']},
        'official_leaderboard_score': None,
        'general_superiority_established': False,
        'interpretation': 'Development outcomes and repeated trials do not establish held-out general superiority.',
    }


def run_development(data_dir: Path, model_path: Path, run_root: Path, plan: dict) -> dict:
    # The notebook embeds the audited official_smoke.run definition first.
    from competition.official_smoke import run
    return execute_plan(run, data_dir, model_path, run_root, plan)


def execute_plan(runner, data_dir: Path, model_path: Path, run_root: Path, plan: dict,
                 *, checkpoint_exporter=None, session_started_monotonic=None) -> dict:
    destination = run_root / 'development_comparison.json'
    if destination.exists():
        raise FileExistsError('A comparison already exists in this run; preserve it and choose a new run directory')
    summarize(plan, [])  # Validate the entire schedule before invoking a model.
    walltime_budget = plan.get('notebook_walltime_budget_seconds')
    if walltime_budget is not None:
        if type(walltime_budget) is not int or walltime_budget <= 0 or session_started_monotonic is None:
            raise ValueError('A declared notebook walltime allowance requires its pre-setup timer')
    if hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Official task revision differs from the locked plan')
    for arm, identity in plan['arms'].items():
        agent_dir = run_root / 'agents' / arm
        if any(p.is_symlink() or not p.resolve().is_relative_to(run_root.resolve())
               for p in agent_dir.rglob('*')):
            raise ValueError('Agent configuration contains a link or escaped path: ' + arm)
        observed = {p.relative_to(agent_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in agent_dir.rglob('*') if p.is_file()}
        if observed != identity['files']:
            raise ValueError('Agent configuration differs from locked plan: ' + arm)
    observations, runtime = [], {}
    run_state = {'stopping_reason': None, 'notebook_walltime_budget_seconds': walltime_budget,
                 'session_timer_includes_setup': session_started_monotonic is not None}
    def save():
        report = {'created_utc': datetime.now(timezone.utc).isoformat(),
                  'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
                  'observations': observations, **run_state, **summarize(plan, observations)}
        atomic_json(destination, report)
        return report
    save()
    if checkpoint_exporter is not None:
        checkpoint_exporter(run_root)
    for index, assignment in enumerate(plan['schedule']):
        arm, task_id = assignment['arm'], assignment['task_id']
        if walltime_budget is not None:
            remaining = session_started_monotonic + walltime_budget - time.monotonic()
            limits = plan['limits']
            required = limits['max_time_minutes'] * 60 + limits['timeout_seconds']
            required += 1200 if not runtime else 0  # Recorded server startup allowance.
            if remaining < required:
                run_state['stopping_reason'] = 'walltime_allowance_insufficient_for_next_assignment'
                return save()  # Never relabel the untouched slots as attempted model failures.
        observation = {'arm': arm, 'task_id': task_id,
                       'result': {'status': 'started',
                                  'started_utc': datetime.now(timezone.utc).isoformat(),
                                  'model_inference_executed': None}}
        if 'trial_id' in assignment:
            observation['trial_id'] = assignment['trial_id']
        observations.append(observation)
        save()
        print(f'DEVELOPMENT_START {index + 1}/{len(plan["schedule"])} {arm} {task_id}', flush=True)
        try:
            if checkpoint_exporter is not None:
                checkpoint_exporter(run_root)
            runner_kwargs = {'runtime': runtime}
            if 'limits' in plan:
                runner_kwargs['task_limits'] = plan['limits']
            result = runner(data_dir, model_path, run_root / 'agents' / arm,
                            run_root / 'results' / f'{index:02d}_{arm}_{task_id}', task_id,
                            **runner_kwargs)
            observation['result'] = result
            report = save()
            print('DEVELOPMENT_RESULT ' + json.dumps(observations[-1]), flush=True)
        except Exception as error:
            observation['result'] = {'status': 'infrastructure_exception',
                                     'error_type': type(error).__name__, 'error': str(error)}
            save()
            if checkpoint_exporter is not None:
                checkpoint_exporter(run_root)
            # Preserve unattempted slots; do not keep generating on a broken backend.
            raise
        if checkpoint_exporter is not None:
            checkpoint_exporter(run_root)
    return report


In [ ]:
# Separately declared CPU26-qualified scoring-compaction transfer. GPU25 is preserved.
"""One real development issue using the organizer's published starter API.

API provenance: Ryan Holbrook, Getting Started - Gemma 4 Developer Agent,
version 2 (353282641), accessed 2026-10-03, Apache-2.0.
https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent
No substitute model, mock grader or inferred leaderboard score is provided.
"""
from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path
import time


def validated_task_limits(value=None):
    defaults = {'timeout_seconds': 300, 'max_time_minutes': 30,
                'max_tool_calls': 50, 'max_turns': 100}
    if value is None:
        return defaults
    if (not isinstance(value, dict) or set(value) != set(defaults)
            or any(type(number) is not int or number <= 0 for number in value.values())):
        raise ValueError('Task limits must declare all four positive integer budgets')
    return dict(value)


def exact_budget_sidecar(agent_dir, task_limits):
    # Check the frozen canonical bytes, without pretending to execute Stage1.
    names = ('timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns')
    expected = ('evaluation:\n' + ''.join('  ' + key + ': ' + str(task_limits[key]) + '\n' for key in names)).encode()
    actual = (Path(agent_dir) / 'eval_config.yaml').read_bytes()
    if actual != expected:
        raise ValueError('Budget sidecar bytes differ from explicit frozen task limits')
    return {'sha256': hashlib.sha256(actual).hexdigest(), 'bytes': len(actual),
            'canonical_sidecar_bytes_match_explicit_limits': True,
            'official_stage1_sidecar_loader_executed': False}


def validate_no_adapter_source_members(agent_dir):
    """Permit exactly the frozen plain baseline or prospectively frozen native-interface or sequential hypothesis package."""
    agent_dir = Path(agent_dir)
    paths = list(agent_dir.rglob('*'))
    if any(path.is_symlink() or not path.resolve().is_relative_to(agent_dir.resolve()) for path in paths):
        raise ValueError('Agent package contains a link or escaped path')
    required = {'agent.yaml', 'prompts/system.md', 'configs/sampling.yaml', 'eval_config.yaml'}
    skill = {'skills/source-import/SKILL.md', 'skills/source-import/scripts/source_import_guard.py'}
    pipeline = {'planner.yaml', 'writer.yaml', 'prompts/planner.md', 'configs/planner_sampling.yaml'}
    actual = {path.relative_to(agent_dir).as_posix() for path in paths if path.is_file()}
    if actual not in (required | skill, required | skill | pipeline):
        raise ValueError('Unexpected files in the frozen no-adapter hypothesis-pipeline study')
    return actual



def compiled_agent_tree(agent):
    row = dict(name=agent.name, agent_class=type(agent).__module__ + '.' + type(agent).__name__)
    if hasattr(agent, 'generate_content_config'):
        row.update(generation=agent.generate_content_config.model_dump(mode='json', exclude_none=True),
            tool_names=[getattr(t, 'name', getattr(t, '__name__', None)) for t in agent.tools],
            instruction_sha256=hashlib.sha256(agent.instruction.encode()).hexdigest(),
            include_contents=agent.include_contents, output_key=agent.output_key)
    row['children'] = [compiled_agent_tree(child) for child in agent.sub_agents]
    return row


def verify_compiled_hypothesis_package(tree, agent_dir):
    agent_dir = Path(agent_dir)
    # These files are frozen and verified against the study plan before runtime.
    import yaml
    expected_writer = yaml.safe_load((agent_dir / 'configs/sampling.yaml').read_bytes())
    native_tools = ['run_command','read_file','edit_file','write_file','get_status','submit_patch',
                    'search_similar_code','get_code_neighbors','get_code_subgraph',None]
    if (agent_dir / 'planner.yaml').exists():
        if not tree['agent_class'].endswith('.SequentialAgent') or len(tree['children']) != 2:
            raise ValueError('Native sequential structure differs')
        planner, writer = tree['children']
        expected_planner = yaml.safe_load((agent_dir / 'configs/planner_sampling.yaml').read_bytes())
        if (planner['generation'] != expected_planner or planner['tool_names'] != []
                or planner['include_contents'] != 'none' or planner['output_key'] != 'repair_hypothesis'
                or planner['instruction_sha256'] != hashlib.sha256((agent_dir / 'prompts/planner.md').read_text(encoding='utf-8').encode()).hexdigest()):
            raise ValueError('Native planner configuration differs')
    else:
        writer = tree
    if (writer['generation'] != expected_writer or writer['tool_names'] != native_tools
            or writer['instruction_sha256'] != hashlib.sha256((agent_dir / 'prompts/system.md').read_text(encoding='utf-8').encode()).hexdigest()):
        raise ValueError('Native writer configuration differs')


def run_scoring_hypothesis_pipeline(data_dir: Path, model_path: Path, agent_dir: Path, results_dir: Path, task_id: str,
        *, runtime: dict | None = None, task_limits: dict | None = None) -> dict:
    task_limits = validated_task_limits(task_limits)
    sidecar_receipt = exact_budget_sidecar(agent_dir, task_limits)
    # Imports and API names below are from the organizer starter, not guesses.
    import asyncio
    import concurrent.futures
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    from swegemma.models.discovery import validate_single_declared_model

    model = 'gemma-4-31b-it-qat-w4a16-ct'
    if not model_path.is_dir() or not (data_dir / 'snapshots').is_dir():
        raise ValueError('Attach the exact required model and complete official data')
    if not torch.cuda.is_available() or torch.cuda.device_count() != 4:
        raise ValueError('This smoke study requires the competition L4 x4 environment')
    gpu_names = [torch.cuda.get_device_name(i) for i in range(4)]
    if any('L4' not in name for name in gpu_names):
        raise ValueError('Unexpected GPU environment; no model/backend substitution')
    tasks = [task for task in load_tasks(data_dir / 'tasks.jsonl') if task.instance_id == task_id]
    if len(tasks) != 1: raise ValueError('Select exactly one real development task')
    declared = validate_single_declared_model(agent_dir)
    if declared != model: raise ValueError('Declared base model differs from competition requirement')
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    # discover_adapters returns a manifest object, whose truth value does not
    # establish whether it contains adapters. This baseline is confined to the
    # exact plan-bound declarative files and optional audited source-import skill, so no adapter weights can enter the run.
    actual_files = validate_no_adapter_source_members(agent_dir)
    results_dir.mkdir(parents=True, exist_ok=False)
    report = {'scope': 'single_public_development_issue_not_leaderboard_or_mobile_result',
              'created_utc': datetime.now(timezone.utc).isoformat(), 'assigned_ids': [task_id],
              'model': model, 'model_path': str(model_path), 'gpu_names': gpu_names,
              'evaluation_limits': task_limits,
              'guide_sha256': hashlib.sha256((data_dir / 'HARNESS_README.md').read_bytes()).hexdigest(),
              'source_sha256': hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest(),
              'packages': {name: importlib.metadata.version(name) for name in
                           ('swegemma', 'adk-submission', 'adk-eval-core', 'vllm')},
              'assigned_count': 1, 'verified_resolved': 0, 'status': 'started',
              'official_leaderboard_score': None}
    report_path = results_dir / 'smoke_outcome.json'
    def save(): report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    save()
    started = time.monotonic()
    try:
        litellm.drop_params = True
        runtime_was_reused = bool(runtime)
        runtime_setup_started = time.monotonic()
        if runtime is not None and runtime:
            if runtime['model_path'] != str(model_path):
                raise ValueError('Cannot reuse a runtime with another model')
            models = runtime['models']
        else:
            server = VllmServer(VllmConfig(model=str(model_path), port=8000, host='127.0.0.1',
            tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
            dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
            gpu_memory_utilization=.80, enable_auto_tool_choice=True, enable_lora=True,
            max_loras=8, max_lora_rank=128, tensor_parallel_size=4, startup_timeout=1200),
            adapter_manifest=adapters)
            server.start()
            models = server.create_model_registry(aliases=[declared], model_prefix='openai/', api_key='EMPTY')
            if runtime is not None:
                runtime.update(model_path=str(model_path), models=models, server=server)
        report['model_runtime_reused'] = runtime_was_reused
        report['model_runtime_setup_wall_seconds'] = time.monotonic() - runtime_setup_started
        limits, constraints = build_submission_limits()
        config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl', snapshots_dir=data_dir / 'snapshots',
            results_dir=results_dir, submission_dir=agent_dir, models=models, sandbox='subprocess',
            **task_limits,
            limits=limits, generation_constraints=constraints, adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(compaction_interval=5, overlap_size=2,
                token_threshold=14336, event_retention_size=5),
            graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'),
            wheels_dir=data_dir / 'wheels', verbose=False)
        effective = {
            'timeout_seconds': config.harness.command_timeout_seconds,
            'max_time_minutes': config.budget.time_minutes,
            'max_tool_calls': config.budget.tool_calls,
            'max_turns': config.budget.turns,
        }
        if effective != task_limits:
            raise ValueError('Actual constructed task budget differs from the frozen plan')
        compaction = {key: getattr(config.events_compaction_config, key) for key in
            ('compaction_interval', 'overlap_size', 'token_threshold', 'event_retention_size')}
        cache = {key: getattr(config.context_cache_config, key) for key in
            ('min_tokens', 'ttl_seconds', 'cache_intervals')}
        if compaction != dict(compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5):
            raise ValueError('Actual compaction differs from the shared frozen setting')
        report['effective_configuration'] = {
            'task_limits': effective,
            'events_compaction': compaction,
            'context_cache': cache,
            'stage1_sidecar_loader_executed': False,
            'budget_entry_path': 'Explicit EvalConfig kwargs after canonical sidecar byte check; Stage1 loader not executed',
            'local_exact_sidecar_verification': sidecar_receipt,
            'timeout_scope': 'Per sandbox command including verifier pytest; no whole-verifier deadline',
        }
        save()
        evaluator = Evaluator(config)
        # A worker owns its asyncio loop, including inside a live notebook kernel.
        compilation = []
        def evaluate():
            # Read-only observation: return the original compiler's same object.
            # No state, callback, model client or tool behavior is replaced.
            from unittest.mock import patch
            import swegemma.harness.agent_runner as native_runner
            original_compile = native_runner.compile_submission
            def observe_compile(*args, **kwargs):
                agent = original_compile(*args, **kwargs)
                tree = compiled_agent_tree(agent)
                verify_compiled_hypothesis_package(tree, agent_dir)
                compilation.append(tree)
                report['native_compilation'] = compilation
                report['native_compilation_observation_changes_agent_behavior'] = False
                save()
                return agent
            with patch.object(native_runner, 'compile_submission', observe_compile):
                return asyncio.run(evaluator.evaluate_task(task=tasks[0], task_index=1, total_tasks=1))
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            result = pool.submit(evaluate).result()
        patch = result.agent_patch or ''
        (results_dir / 'generated.patch').write_text(patch, encoding='utf-8')
        (results_dir / 'test_output.txt').write_text(result.test_output or '', encoding='utf-8')
        # TaskResult is the organizer's Pydantic result type. Preserve grader
        # diagnostics rather than retaining only an exit code and losing cause.
        (results_dir / 'official_task_result.json').write_text(
            json.dumps(result.model_dump(mode='json', exclude={'trace'}), indent=2) + '\n', encoding='utf-8')
        report.update(status='completed', official_resolved=result.resolved,
                      test_exit_code=result.test_exit_code, tool_calls=result.tool_calls,
                      task_evaluation_duration_seconds=result.duration_seconds, patch_bytes=len(patch.encode()),
                      patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),
                      verified_resolved=int(result.resolved is True and result.test_exit_code == 0 and bool(patch)))
    except Exception as error:
        report.update(status='failed', error_type=type(error).__name__, error=str(error))
        raise
    finally:
        report['setup_and_evaluation_wall_seconds'] = time.monotonic() - started
        save()
    return report


In [ ]:
"""Declared receipt-aware stopping, retaining all assigned trial denominators."""
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import time



def ordinary_agent_timeout(result_dir, result, limits):
    """Recognize only the observed official timeout contract; unknown -1 halts."""
    path = result_dir / 'official_task_result.json'
    if not path.is_file():
        return False
    official = json.loads(path.read_bytes())
    message = f"Agent exceeded session timeout ({limits['max_time_minutes']} min)"
    return (official.get('status') == 'SUCCESS' and official.get('resolved') is False
            and official.get('test_exit_code') == -1 and official.get('agent_patch') == ''
            and official.get('error_message') == message
            and result.get('status') == 'completed' and result.get('official_resolved') is False
            and result.get('patch_bytes') == 0)


def execute_plan_with_arm_limits(runner, data_dir: Path, model_path: Path, run_root: Path, plan: dict,
                         *, checkpoint_exporter=None, session_started_monotonic=None) -> dict:
    destination = run_root / 'development_comparison.json'
    if destination.exists():
        raise FileExistsError('Preserve the previous comparison; use a new run directory')
    summarize(plan, [])
    if set(plan['arm_limits']) != set(plan['arms']):
        raise ValueError('Every arm requires exactly one explicit limit record')
    expected_keys = {'timeout_seconds', 'max_time_minutes', 'max_tool_calls', 'max_turns'}
    for limits in plan['arm_limits'].values():
        if set(limits) != expected_keys or any(type(v) is not int or v <= 0 for v in limits.values()):
            raise ValueError('Invalid per-arm limits')
        if any(limits[k] > plan['limits'][k] for k in expected_keys):
            raise ValueError('Per-arm limits exceed the declared walltime envelope')
    walltime_budget = plan.get('notebook_walltime_budget_seconds')
    if walltime_budget is not None and (type(walltime_budget) is not int or walltime_budget <= 0
                                      or session_started_monotonic is None):
        raise ValueError('A notebook allowance requires a positive budget and pre-setup timer')
    if hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Official task revision differs from the locked plan')
    for arm, identity in plan['arms'].items():
        agent_dir = run_root / 'agents' / arm
        if any(p.is_symlink() or not p.resolve().is_relative_to(run_root.resolve())
               for p in agent_dir.rglob('*')):
            raise ValueError('Agent configuration contains a link or escaped path: ' + arm)
        observed = {p.relative_to(agent_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in agent_dir.rglob('*') if p.is_file()}
        if observed != identity['files']:
            raise ValueError('Agent configuration differs from the locked plan: ' + arm)
    observations, runtime = [], {}
    state = {'stopping_reason': None, 'notebook_walltime_budget_seconds': walltime_budget,
             'session_timer_includes_setup': session_started_monotonic is not None,
             'unknown_returned_official_minus_one_is_infrastructure_stop': True,
             'ordinary_agent_timeouts_retained_without_stopping_remaining_assignments': True}
    def save():
        report = {'created_utc': datetime.now(timezone.utc).isoformat(),
                  'plan_sha256': hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
                  'observations': observations, **state, **summarize(plan, observations)}
        atomic_json(destination, report)
        return report
    def checkpoint():
        if checkpoint_exporter is not None:
            checkpoint_exporter(run_root)
    save()
    checkpoint()
    for index, assignment in enumerate(plan['schedule']):
        arm, task_id = assignment['arm'], assignment['task_id']
        if walltime_budget is not None:
            remaining = session_started_monotonic + walltime_budget - time.monotonic()
            limits = plan['arm_limits'][arm]
            required = limits['max_time_minutes'] * 60 + limits['timeout_seconds']
            required += 1200 if not runtime else 0
            if remaining < required:
                state['stopping_reason'] = 'walltime_allowance_insufficient_for_next_assignment'
                report = save()
                checkpoint()
                return report
        result_dir = run_root / 'results' / f'{index:02d}_{arm}_{task_id}'
        row = {'arm': arm, 'task_id': task_id,
               'result': {'status': 'started', 'started_utc': datetime.now(timezone.utc).isoformat(),
                          'model_inference_executed': None}}
        if 'trial_id' in assignment:
            row['trial_id'] = assignment['trial_id']
        observations.append(row)
        save()
        print(f'DEVELOPMENT_START {index + 1}/{len(plan["schedule"])} {arm} {task_id}', flush=True)
        try:
            checkpoint()
            kwargs = {'runtime': runtime}
            kwargs['task_limits'] = plan['arm_limits'][arm]
            row['result'] = runner(data_dir, model_path, run_root / 'agents' / arm,
                                   result_dir, task_id, **kwargs)
        except Exception as error:
            row['result'] = {'status': 'infrastructure_exception',
                             'error_type': type(error).__name__, 'error': str(error)}
            state.update(stopping_reason='raised_infrastructure_exception',
                         stopped_after_assignment=dict(assignment))
            save()
            checkpoint()
            raise
        if row['result'].get('test_exit_code') == -1:
            if ordinary_agent_timeout(result_dir, row['result'], plan['arm_limits'][arm]):
                row['result']['outcome_classification'] = 'agent_timeout_without_graded_patch'
            else:
                # Preserve unknown grading/setup failures and stop safely.
                state.update(stopping_reason='returned_official_test_exit_minus_one',
                         stopped_after_assignment=dict(assignment),
                         stopped_official_receipt_path=(result_dir / 'official_task_result.json').relative_to(run_root).as_posix(),
                         stopped_smoke_receipt_path=(result_dir / 'smoke_outcome.json').relative_to(run_root).as_posix())
        report = save()
        print('DEVELOPMENT_RESULT ' + json.dumps(row), flush=True)
        # The failure receipt and assigned/unattempted counts are durable before halt.
        checkpoint()
        if state['stopping_reason'] is not None:
            return report
    return report


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result = execute_plan_with_arm_limits(run_scoring_hypothesis_pipeline, DATA_DIR, MODEL_PATH, RUN, PLAN,
        checkpoint_exporter=export_run, session_started_monotonic=SESSION_STARTED_MONOTONIC)
    print('BROADER_SCORING_COMPARISON=' + json.dumps(result))
else:
    print('Definitions only; no installation, inference or official grading')


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    export_run(RUN)
else:
    print('No worker evidence exported by local definition check')
